# Import Library

In [ ]:
%pip install pdfplumber pandas
%pip install openpyxl==3.1.2
%pip install pykalman
%pip install google-colab

import pdfplumber
import glob
import pandas as pd
import os
import re
import csv
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print("Running locally — skipping Google Colab mounting.")

from collections import defaultdict
from pykalman import KalmanFilter

import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: openpyxl==3.1.2 in c:\users\mingr\appdata\local\programs\python\python313\lib\site-packages (3.1.2)




[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Running locally — skipping Google Colab mounting.


ERROR: Could not find a version that satisfies the requirement google-colab (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for google-colab


# Data extraction from pdf into csv


In [ ]:
# Folder containing your PDFs
pdf_folder = "/content/drive/My Drive/FYP_Fish/Fish Landings Pdf Datasets"
output_base = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"
os.makedirs(output_base, exist_ok=True)

In [ ]:
# Regex pattern to detect "Jadual ( Table ) xxx"
jadual_pattern = r"Jadual\s*\(\s*Table\s*\)\s*\d+(\.\d+)*"
sambungan_pattern = r'(Samb\. Dari Jadual|Cont\'d From)'  # Detect continuation pages

In [ ]:
def process_pdf_files(pdf_folder, output_base):
    """
    Extract tables from PDFs, naming files based on 'Jadual' title if available, otherwise by page number.
    Merge continuation tables ('Sambungan') into previous CSV.
    """
    for pdf_file in os.listdir(pdf_folder):
        if pdf_file.lower().endswith(".pdf"):
            pdf_path = os.path.join(pdf_folder, pdf_file)
            pdf_name = os.path.splitext(pdf_file)[0]
            output_dir = os.path.join(output_base, pdf_name)
            os.makedirs(output_dir, exist_ok=True)

            print(f"\n📄 Processing PDF: {pdf_file}")

            previous_csv_path = None  # 🔥 To track the last valid CSV for sambungan

            with pdfplumber.open(pdf_path) as pdf:
                for page_num, page in enumerate(pdf.pages, start=1):
                    text = page.extract_text()

                    # Default filename if no 'Jadual' or 'Sambungan' detected
                    filename = f"Jadual_Page_{page_num}.csv"

                    new_table_found = False

                    if text:
                        if re.search(jadual_pattern, text, re.IGNORECASE):
                            # New table found
                            header = text.split("\n")[0].strip()
                            header_sanitized = re.sub(r'[\\/*?:"<>|]', "", header)

                            if header_sanitized.lower().startswith('jadual'):
                                filename = f"{header_sanitized}.csv"
                            new_table_found = True

                        elif re.search(sambungan_pattern, text, re.IGNORECASE):
                            # 🔥 This page is continuation ('Sambungan')
                            if previous_csv_path:
                                filename = os.path.basename(previous_csv_path)
                                print(f"  🔗 Detected continuation on page {page_num}, merging into {filename}")
                            else:
                                # No previous table to continue, fallback to default
                                filename = f"Jadual_Page_{page_num}.csv"
                        else:
                            # No 'Jadual' and no 'Sambungan' detected
                            pass

                    current_csv_path = os.path.join(output_dir, filename)
                    os.makedirs(os.path.dirname(current_csv_path), exist_ok=True)

                    if new_table_found:
                        print(f"  🔍 Saving NEW table from page {page_num} as {filename}")
                        previous_csv_path = current_csv_path  # Update the previous CSV path
                    elif previous_csv_path and filename == os.path.basename(previous_csv_path):
                        # Sambungan pages (merging to previous)
                        pass
                    else:
                        print(f"  🔍 Saving table from page {page_num} as {filename}")
                        previous_csv_path = current_csv_path  # Reset because no sambungan detected

                    # Extract tables
                    tables = page.extract_tables()
                    first_write = not os.path.exists(current_csv_path)  # Only write header if file does not exist yet

                    for table in tables:
                        if table:
                            df = pd.DataFrame(table)
                            df.to_csv(current_csv_path, mode='a', index=False, header=first_write)
                            first_write = False  # After first write, set header=False for next tables

            print(f"✅ Finished PDF: {pdf_file}")

            num_files = len([f for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))])
            print(f"Number of files in {output_dir}: {num_files}")

    print("\n🎉 All PDFs processed successfully!")

In [ ]:
# process_pdf_files(pdf_folder, output_base)

## Handle 4.5 related Datasets

### Handle 4.5 datasets
Merge 4.5 datasets into a dataset and rename it to 4.5 year and saved into a folder named 4.5

In [ ]:
# Create a folder for saving merged dataset if it doesn't exist
output_base = "/content/drive/My Drive/FYP_Fish/4.5"

os.makedirs(output_base, exist_ok=True)

In [ ]:
# Define the base directory where the datasets are stored
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

# Define the paths for each dataset for each year
file_paths = {
    "2008": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2008', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2008', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2008', 'Jadual_Page_9.csv')
    ],
    "2009": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2009', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2009', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2009', 'Jadual_Page_27.csv')
    ],
    "2010": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2010', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2010', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2010', 'Jadual_Page_9.csv')
    ],
    "2011": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2011', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2011', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2011', 'Jadual_Page_9.csv')
    ],
    "2012": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2012', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2012', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2012', 'Jadual_Page_27.csv')
    ],
    "2013": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2013', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2013', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2013', 'Jadual_Page_27.csv')
    ],
    "2014": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2014', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2014', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2014', 'Jadual_Page_27.csv')
    ],
    "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_9.csv')
    ],
    "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_9.csv')
    ],
    "2017": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2017', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2017', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2017', 'Jadual_Page_9.csv')
    ],
    "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_9.csv')
    ],
    "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_9.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_8.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_9.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)

    # Create a filename for the year and save it
    output_file = os.path.join(output_base, f'4.5 - {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2015.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2016.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2017.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2023.csv


In [ ]:
# Create a folder for saving merged dataset if it doesn't exist
output_base = "/content/drive/My Drive/FYP_Fish/4.5"

os.makedirs(output_base, exist_ok=True)

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)

    # Create a filename for the year and save it
    output_file = os.path.join(output_base, f'4.5 - {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2015.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2016.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2017.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5/4.5 - 2023.csv


### Handle 4.4 datasets  
Extract all 4.4 datasets and saved to a folder named 4.4

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.4"
os.makedirs(output_base, exist_ok=True)

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.4.csv"
        if file == "Jadual (Table) 4.4.csv" or file == "Jadual(Table) 4.4.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Skip folders containing "2015"
            if '2015' in year:
                continue

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path)

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.4 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2007.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Drive/

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2015": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.xlsx",
    "2018": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.4"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.4 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2021.csv


In [ ]:
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

file_paths = {
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_6.csv')
     ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(output_base, f'4.4 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2016
Total rows before merging: 168
Total rows after merging: 168
Saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv


### Handle 4.5.2 Dataset - West Coast of Peninsular Malaysia
(Perlis, Kedah, Pahang, Perak, Selangor, Negeri Sembilan, Malacca, West Johor)

Extract all 4.5.2 datasets and saved to a folder named 4.5.2 West

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.5.2 West"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_15.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_15.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_15.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_15.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(output_base, f'4.5.2 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")



 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1528
Total rows after merging: 1528
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1544
Total rows after merging: 1544
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.5.2.csv"
        if file == "Jadual (Table) 4.5.2.csv" or file == "Jadual(Table) 4.5.2.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.5.2 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2017.csv
🔍 Saved /content/drive/My Driv

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.5.2 West"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.5.2 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv


### Handle 4.5.3 Dataset - East Coast of Peninsular Malaysia
(Kelantan, Terengganu, Pahang, East Johor)
Extract all 4.5.3 datasets and saved to a folder named 4.5.3 East

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.5.3 East"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"


In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.5.4.csv"
        if file == "Jadual (Table) 4.5.3.csv" or file == "Jadual(Table) 4.5.3.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.5.3 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2007.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Driv

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.5.3 East"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.5.3 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2021.csv


### Handle 4.5.4 Dataset - Sarawak
Extract all 4.5.4 datasets and saved to a folder named 4.5.4 Sarawak

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"


In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.5.4.csv"
        if file == "Jadual (Table) 4.5.4.csv" or file == "Jadual(Table) 4.5.4.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.5.4 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2017.csv


Transform 2018, 2019 and 2021 data from xlsx into csv



In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.5.4 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2021.csv


### Handle 4.5.5 Dataset - Sabah
Extract all 4.5.5 datasets and saved to a folder named 4.5.5 Sabah



In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.5.5.csv"
        if file == "Jadual (Table) 4.5.5.csv" or file == "Jadual(Table) 4.5.5.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.5.5 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv
🔍 Saved /content/dri

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.5.5 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2021.csv


### Handle 4.5.6 Dataset - Labuan
Extract all 4.5.6 datasets and saved to a folder named 4.5.6 Labuan



In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.5.5.csv"
        if file == "Jadual (Table) 4.5.6.csv" or file == "Jadual(Table) 4.5.6.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.5.6 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2007.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /c

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.5.6 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2021.csv


## Handle 4.6 related Datasets

### Handle 4.6.2 Dataset - West Coast of Peninsular Malaysia

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.6.2 West"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.6.2.csv"
        if file == "Jadual (Table) 4.6.2.csv" or file == "Jadual(Table) 4.6.2.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.6.2 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Driv

Transform 2018, 2019 and 2021 data from xlsx into csv


In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.xlsx",
    "2022": "/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2022.xlsx",
    "2023": "/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2023.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.6.2 West"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.6.2 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv


### Handle 4.6.3 Dataset - East Coast of Peninsular Malaysia

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.6.3 East"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.6.3.csv"
        if file == "Jadual (Table) 4.6.3.csv" or file == "Jadual(Table) 4.6.3.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.6.3 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/drive/My Driv

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2021.xlsx",
    "2022": "/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2022.xlsx",
    "2023": "/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.6.3 East"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.6.3 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2021.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv


### Handle 4.6.4 Dataset - Sarawak

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.6.4.csv"
        if file == "Jadual (Table) 4.6.4.csv" or file == "Jadual(Table) 4.6.4.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.6.4 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv


Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2021.xlsx",
    "2022": "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2022.xlsx",
    "2023": "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.6.4 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv


### Handle 4.6.5 Dataset - Sabah

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.6.5.csv"
        if file == "Jadual (Table) 4.6.5.csv" or file == "Jadual(Table) 4.6.5.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.6.5 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /content/dri

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2018": "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2021.xlsx",
    "2022": "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.xlsx",
    "2023": "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2023.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.6.5 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2021.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv


### Handle 4.6.6 Dataset - Labuan

In [ ]:
# Define the base directory where the datasets are stored
output_base = "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan"
os.makedirs(output_base, exist_ok=True)
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

In [ ]:
# Walk through the directories and files
for root, dirs, files in os.walk(base_dir):
    for file in files:
        # Check if the file name matches "Jadual (Table) 4.6.6.csv"
        if file == "Jadual (Table) 4.6.6.csv" or file == "Jadual(Table) 4.6.6.csv":
            # Extract the year from the folder name (assume the folder name is the year)
            year = os.path.basename(root)

            # Construct the full file path
            file_path = os.path.join(root, file)

            # Read the CSV file
            df = pd.read_csv(file_path, on_bad_lines='skip')

            # Define the output file path for the current year
            output_file = os.path.join(output_base, f'4.6.6 - {year}.csv')

            # Save the dataframe to the new CSV file
            df.to_csv(output_file, index=False)

            print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2010.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2006.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2009.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2008.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv
🔍 Saved /c

Transform 2018, 2019 and 2021 data from xlsx into csv

In [ ]:
xlsx_files = {
    "2019": "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2021.xlsx",
    "2022": "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2022.xlsx",
    "2023": "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.xlsx"
}

output_base = "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan"

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(output_base, f'4.6.6 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2022.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv


# Extract Species that is most consume by Malaysian

Based on Fish consumption pattern among adults of different ethnics in Peninsular Malaysia by National Library of Medicine, we extract the fish species that has most consumes from dataset 4.5.2, 4.4.3, 4.5.4 and 4.5.5.  

---
**Marine Fish**
1. Bilis
2. Pelata
3. Selar
4. Selar Kuning
5. Kayu/Tongkol/Aya Hitam
6. Kayu/Tongkol/Aya Kurik
7. Kayu/Tongkol/Aya Selasih
8. Kayu/Tongkol/Aya Jalur
9. Kayu/Tongkol/Aya Peluru
10. Selayang/Curut
11. Bawal Hitam
12. Bawal Putih
13. Bawal Bujang
14. Bawal Tambak
15. Bawal Selatan
16. Tenggiri
17. Kerisi
18. Kerisi Bali
19. Gelama/Tengkerong
20. Duri/Pelutan/Utik
21. Siakap
22. Jenahak Besar
23. Jenahak Sederhana
24. Jenahak Kecil
25. Terubuk
26. Terubuk T.Macrura
27. Parang-Parang
28. Talang
29. Senangin
30. Senangin Buis
31. Belanak
32. Gerut-Gerut Besar
33. Gerut-Gerut Sederhana
34. Gerut-Gerut Kecil
35. Selangat
36. Rambai/Ebek
37. Yu
38. Kerapu Besar
39. Kerapu Sederhana
40. Kerapu Kecil
41. Puput
42. Sebelah
43. Biji Nangka
44. Biji Nangka India
45. Biji Nangka Tanda Merah
46. Jolong-Jolong
47. Alu-Alu/Kacang-Kacang

---

**Marine Crab**
1. Ketam Laut
2. Ketam Renjong
3. Ketam Batu
4. Ketam Nipah

---

**Marine Prawn**
1. Udang Harimau
2. Udang Putih Besar/Kertas
3. Udang Putih Sedang
4. Udang Putih Kecil/Kertas
5. Udang Kuning
6. Udang Merah
7. Udang Baring
8. Udang Susu

---

**Marine Sotong**
1. Sotong Katak
2. Sotong Kurita
3. Sotong Mengabang
4. Sotong Jarum
5. Sotong Torak
6. Sotong Daun


## Functions Code Snippet

In [ ]:
# List of species to extract
species_list = [
    # Marine Fish
    "Bilis/Bunga Air", "Pelata", "Selar", "SelarKuning", "Kayu/Tongkol/AyaHitam",
    "Kayu/Tongkol/AyaKurik", "Kayu/Tongkol/Aya Selasih", "Kayu/Tongkol/AyaJalur",
    "Kayu/Tongkol/AyaPeluru", "Selayang/Curut", "BawalHitam", "BawalPutih",
    "BawalBujang", "BawalTambak", "BawalSelatan", "Tenggiri", "Kerisi",
    "KerisiBali", "Gelama/Tengkerong", "Duri/Pelutan/Utik", "Siakap", "JenahakBesar",
    "JenahakSederhana", "JenahakKecil", "Terubuk", "TerubukT.Macrura", "Parang-Parang",
    "Talang", "Senangin", "SenanginBuis", "Belanak", "Gerut-GerutBesar", "Gerut-GerutSederhana",
    "Gerut-GerutKecil", "Selangat", "Rambai/Ebek", "Yu", "KerapuBesar", "KerapuSederhana",
    "KerapuKecil", "Puput", "Sebelah", "BijiNangka", "BijiNangkaIndia", "BijiNangkaTandaMerah",
    "Jolong-Jolong", "Alu-Alu/Kacang-Kacang",

    # Marine Crab
    "KetamLaut", "KetamRenjong", "KetamBatu", "KetamNipah",

    # Marine Prawn
    "UdangHarimau", "UdangPutih Besar/Kertas", "UdangPutihSedang", "UdangPutihKecil/Kertas",
    "UdangKuning", "UdangMerah", "UdangBaring", "UdangSusu",

    # Marine Sotong
    "SotongKatak", "SotongKurita", "SotongMengabang", "SotongJarum",
    "SotongTorak", "SotongDaun"
]

In [ ]:
species_list_gear = [
    'Kerisibali', 'Ketamlaut', 'Selar', 'Alu-Alu/Kacang-Kacang',
    'Bijinangka', 'Kerisi', 'Sebelah', 'Gelama/Tengkerong',
    'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Tenggiri', 'Bawalputih',
    'Yu', 'Selarkuning', 'Siakap'
]

In [ ]:
# Create DataFrame and save
columns = ["Species", "Year", "January", "February", "March", "April", "May", "June",
           "July", "August", "September", "October", "November", "December"]

Functions for data cleaning

In [ ]:
# Remove duplicated rows
def clean_csv_files(input_dir):
    """Loop through all CSV files in the directory, clean them by removing duplicates, fixing species names, and normalizing numbers."""
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    # Read CSV
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

                    # Drop duplicate rows
                    df.drop_duplicates(inplace=True)

                    # Remove spaces in 'Species' column (if it exists)
                    first_col = df.columns[0]
                    second_col = df.columns[1]

                    df[first_col] = df[first_col].astype(str).str.replace(' ', '')
                    df[second_col] = df[second_col].astype(str).str.replace(' ', '')

                    # Convert space-separated numbers and comma-separated numbers
                    def clean_numeric(value):
                        if pd.isna(value):
                            return value
                        if isinstance(value, str):
                            value = re.sub(r'(?<=\d) +(?=\d)', '', value)
                            # Remove commas from numbers (e.g., "1,003" -> "1003")
                            value = value.replace(',', '')
                            value = value.replace(' ', '')
                        try:
                            return float(value)
                        except ValueError:
                            return value

                    # Apply to all numeric-looking columns
                    for col in df.columns:
                        df[col] = df[col].apply(clean_numeric)

                    # Save cleaned data back
                    df.to_csv(file_path, index=False, encoding='utf-8')
                    print(f"✅ Cleaned and saved: {file_path}")

                except Exception as e:
                    print(f"❌ Failed to clean {file_path}: {e}")


In [ ]:
def clean_csv_files_gear(input_dir):
    """
    Loop through all CSV files in the directory, clean them by:
    - Removing duplicate rows
    - Fixing formatting in the first two columns (assumed to be species/gear-related)
    - Normalizing numeric values
    NOTE: Does NOT remove any columns.
    """
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

                    # Drop duplicate rows only (columns preserved)
                    df.drop_duplicates(inplace=True)

                    # Clean 'Species' and 'Gear' columns if they exist
                    if df.shape[1] >= 2:
                        df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.replace(' ', '', regex=False)
                        df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)

                    # Function to clean numeric-looking strings without removing columns
                    def clean_numeric(value):
                        if pd.isna(value):
                            return value
                        if isinstance(value, str):
                            value = re.sub(r'(?<=\d) +(?=\d)', '', value)  # Remove internal spaces
                            value = value.replace(',', '')                # Remove commas
                            value = value.replace(' ', '')                # Remove stray spaces
                        try:
                            return float(value)
                        except ValueError:
                            return value  # Keep as-is if not convertible

                    # Apply numeric cleaning without dropping any column
                    for col in df.columns:
                        df[col] = df[col].apply(clean_numeric)

                    # Save cleaned file
                    df.to_csv(file_path, index=False, encoding='utf-8')
                    print(f"✅ Cleaned and saved: {file_path}")

                except Exception as e:
                    print(f"❌ Failed to clean {file_path}: {e}")

In [ ]:
def remove_unnecessary_columns_rows(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith(".csv"):
                continue

            file_path = os.path.join(root, file)
            match = re.search(r'(20\d{2})', file)
            year = int(match.group(0)) if match else None

            if not year:
                print(f"⚠️ Skipping {file} (no year found)")
                continue

            try:
                df = pd.read_csv(file_path, engine='python', on_bad_lines='skip', header=None)

                # Apply different row slicing based on the year
                if year in list(range(2011, 2018)) + [2020]:
                    df = df.iloc[3:, 1:]
                elif year in [2018, 2019]:
                    df = df.iloc[9:, 1:]
                elif year == 2021:
                    df = df.iloc[8:, 1:]
                elif year in [2022, 2023]:
                    df = df.iloc[3:, :]
                else:
                    print(f"ℹ️ No rule for {year} — keeping original")
                    continue

                # Remove rows containing "Jadual 4.5.2" (case-insensitive)
                df = df[~df.apply(lambda row: row.astype(str).str.contains("jadual 4.5.2", case=False).any(), axis=1)]

                # Remove empty columns for specific years (2018, 2019, 2021)
                if year in [2018, 2019, 2021]:
                    df = df.dropna(axis=1, how='all')  # Drop columns with all NaN values

                cleaned_filename = f"{file}"
                save_path = os.path.join(output_dir, cleaned_filename)
                df.to_csv(save_path, index=False, header=False)
                print(f"✅ Saved cleaned file to {save_path}")

            except Exception as e:
                print(f"❌ Error processing {file_path}: {e}")

In [ ]:
def remove_unnecessary_columns_rows_fishing_gear(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith(".csv"):
                continue

            file_path = os.path.join(root, file)
            match = re.search(r'(20\d{2})', file)
            year = int(match.group(0)) if match else None

            if not year:
                print(f"⚠️ Skipping {file} (no year found)")
                continue

            try:
                df = pd.read_csv(file_path, engine='python', on_bad_lines='skip', header=None)

                # Apply different row slicing based on the year
                if year in list(range(2011, 2018)) + [2020]:
                    df = df.iloc[3:, 1:]
                elif year in [2018, 2019]:
                    df = df.iloc[9:, 1:]
                elif year == 2021:
                    df = df.iloc[9:, 1:]
                elif year in [2022, 2023]:
                    df = df.iloc[3:, :]
                else:
                    print(f"ℹ️ No rule for {year} — keeping original")
                    continue

                # Remove rows containing "Jadual 4.5.2" (case-insensitive)
                df = df[~df.apply(lambda row: row.astype(str).str.contains("jadual 4.5.2", case=False).any(), axis=1)]

                # Remove empty columns for specific years (2018, 2019, 2021)
                if year in [2018, 2019, 2021]:
                    df = df.dropna(axis=1, how='all')  # Drop columns with all NaN values

                cleaned_filename = f"{file}"
                save_path = os.path.join(output_dir, cleaned_filename)
                df.to_csv(save_path, index=False, header=False)
                print(f"✅ Saved cleaned file to {save_path}")

            except Exception as e:
                print(f"❌ Error processing {file_path}: {e}")

Check whether the species in the species list appear in the csv

In [ ]:
def find_species_in_csv_files(input_dir, species_list, year_range=(2011, 2023)):
    """
    Walks through CSV files in a directory, identifies files within a specified year range,
    and checks if they contain any of the species in the provided list.

    Args:
        input_dir (str): Path to the directory containing CSV files.
        species_list (list): List of species names to search for.
        year_range (tuple): Range of years (inclusive) to consider.
                             Defaults to (2011, 2023).

    Returns:
        defaultdict: A dictionary where keys are years and values are sets of species found
                     in files for that year.
    """

    found_species_by_year = defaultdict(list)

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv") and not file.startswith("extracted_species"):
                file_path = os.path.join(root, file)
                print(f"\n🔍 Processing {file_path}")

                # Extract year from filename or folder path
                match = re.search(r'(20\d{2}|19\d{2})', file)
                year = match.group(0) if match else None

                if not year:
                    folder_match = re.search(r'(20\d{2}|19\d{2})', root)
                    year = folder_match.group(0) if folder_match else None

                if not year:
                    print(f"⚠️ Skipping {file} — No year found in file or folder")
                    continue

                # Check if year is within the specified range
                if int(year) < year_range[0] or int(year) > year_range[1]:
                    print(f"⚠️ Skipping {file} — Year {year} out of range")
                    continue

                print(f"🔎 Processing {file} — Year {year}")

                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')
                    df.dropna(how='all', inplace=True)
                    df.columns = df.columns.astype(str)  # Ensure column names are strings

                    for species in species_list:
                        # Find rows containing the species in the first column or any relevant column
                        matches = df[df.iloc[:, 0].astype(str).str.contains(species, case=False, na=False)]

                        if not matches.empty:
                            print(f"✅ Found {species} in {file}")
                            found_species_by_year[year].append((species, matches))  # Store species with the matching rows
                        else:
                            print(f"❌ {species} not found in {file}")

                except Exception as e:
                    print(f"❌ Failed to read {file_path}: {e}")

    return found_species_by_year


In [ ]:
# Summarize species that match
def summarize_species_matches(found_species_by_year):
    """
    Summarizes species occurrences by year.

    Args:
        found_species_by_year (dict): Dictionary where keys are years and values are lists of
                                      (species_name, DataFrame of matches) tuples.
    """
    for year in sorted(found_species_by_year):
        matches = found_species_by_year[year]
        species_counts = {}

        for species, _ in matches:
            species_counts[species] = species_counts.get(species, 0) + 1

        print(f"\n📅 Year: {year}")
        print(f"✅ Total unique species found: {len(species_counts)}")

Drop total column which is the last column in each csv

In [ ]:
def drop_last_column(input_dir):
    """
    Cleans each CSV in the directory:
    - Removes empty rows
    - Removes empty columns (columns with all empty values)
    - Drops the last column (optional final total column)
    """
    for filename in os.listdir(input_dir):
        if filename.lower().endswith('.csv'):
            file_path = os.path.join(input_dir, filename)

            try:
                # Read the file (try utf-8 first, fallback to latin1)
                try:
                    with open(file_path, 'r', newline='', encoding='utf-8') as f:
                        rows = list(csv.reader(f))
                except UnicodeDecodeError:
                    with open(file_path, 'r', newline='', encoding='latin1') as f:
                        rows = list(csv.reader(f))

                # Remove empty rows
                rows = [row for row in rows if any(cell.strip() for cell in row)]
                if not rows:
                    print(f"⚠️ Skipped {filename} — File is empty after removing empty rows.")
                    continue

                # Transpose to work with columns, drop empty columns
                transposed = list(zip(*rows))
                non_empty_columns = [col for col in transposed if any(cell.strip() for cell in col)]
                cleaned_rows = list(zip(*non_empty_columns))

                # Drop the last column (if more than 1 column remains)
                if cleaned_rows and len(cleaned_rows[0]) > 1:
                    cleaned_rows = [row[:-1] for row in cleaned_rows]

                # Save the cleaned data
                with open(file_path, 'w', newline='', encoding='utf-8') as f:
                    writer = csv.writer(f)
                    writer.writerows(cleaned_rows)

                print(f"✅ Cleaned {filename} — Empty rows/columns and last column removed")

            except Exception as e:
                print(f"❌ Error processing {filename}: {e}")

Add header for each csv

In [ ]:
 # Define the month columns
months = ['January', 'February', 'March', 'April', 'May', 'June','July', 'August', 'September', 'October', 'November', 'December']

# Define the header row
header = ['Species'] + months

In [ ]:
# Fishing gear header
gear_header = ['Species', 'Trawl Nets', 'Fish Purse Seines', 'Anchovy Purse Seines',
               'Other Seines', 'Drift/Gill Nets', 'Lift Nets', 'Stationary Traps',
               'Portable Traps', 'Hooks & Lines', 'Bag Nets', 'Barrier Nets',
               'Push/Scoop Nets', 'ShellfishCollection', 'Miscellaneous', 'Total']

In [ ]:
# Add a row for the header at the top of each csv file
def add_header(directory_path, header):
    """
    Process all CSV files in a directory: add header row with Species and month columns.
    """
    # Store the results for each processed file
    results = []

    # Loop through files in the directory
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.csv'):
            file_path = os.path.join(directory_path, filename)

            try:
                # Read the existing content of the file
                with open(file_path, 'r', newline='') as file:
                    reader = csv.reader(file)
                    rows = list(reader)

                # Write back with the header row added at the top
                with open(file_path, 'w', newline='') as file:
                    writer = csv.writer(file)
                    # Write the header first
                    writer.writerow(header)
                    # Write all the existing rows from the original file
                    writer.writerows(rows)

                results.append(f"Added header to {file_path}")
            except Exception as e:
                results.append(f"Error processing {file_path}: {e}")

    return results

Add year column based on the csv file name

In [ ]:
def add_year_column(directory_path):
    """
    Process all CSV files in a directory and add a 'Year' column based on the file name.
    The year is extracted from the file name (assuming the year is present in the filename).
    """
    results = []

    # Loop through files in the directory
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.csv'):
            file_path = os.path.join(directory_path, filename)

            try:
                # Extract year from the filename (assuming the year is in the format YYYY)
                match = re.search(r'(20\d{2}|19\d{2})', filename)
                year = match.group(0) if match else "Unknown"

                # Read the existing content of the file
                with open(file_path, 'r', newline='') as file:
                    reader = csv.reader(file)
                    rows = list(reader)

                # Add the 'Year' column to the header if not already present
                if 'Year' not in rows[0]:
                    rows[0].append('Year')  # Append 'Year' column to header

                # Add the year value to each row (skip the header)
                for row in rows[1:]:
                    row.append(year)

                # Write the updated content back to the file
                with open(file_path, 'w', newline='') as file:
                    writer = csv.writer(file)
                    writer.writerows(rows)

                results.append(f"Added 'Year' column to {file_path} with year: {year}")
            except Exception as e:
                results.append(f"Error processing {file_path}: {e}")

    return results

Extract the species in species list from all the csv and based on header coloumn

In [ ]:
def extract_gear_in_csv(input_dir, output_path, year_range=(2011, 2023), species_list=[]):
    """
    Extract rows containing any species in species_list (case-insensitive, partial match)
    from all CSV files in input_dir for years within year_range.
    Matches can be in any column.
    Outputs string fields with only the first letter capitalized (title case).
    """
    results_dict = []

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                match = re.search(r'(20\d{2}|19\d{2})', file)
                if not match:
                    print(f"⚠️ Skipping {file} — No 4-digit year found in filename")
                    continue

                year = int(match.group())
                if year < year_range[0] or year > year_range[1]:
                    print(f"⚠️ Skipping {file} — Year {year} out of range {year_range}")
                    continue

                file_path = os.path.join(root, file)
                print(f"\n📄 Scanning file: {file} (Year: {year})")

                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')
                    df.dropna(how='all', inplace=True)
                    df.columns = df.columns.astype(str)

                    df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

                    match_rows = []

                    for species in species_list:
                        species_lower = species.lower()

                        for col in df_lower.columns:
                            mask = df_lower[col].astype(str).str.contains(species_lower, na=False)
                            matched = df[mask]

                            if not matched.empty:
                                print(f"✅ Found '{species}' in {file} — {len(matched)} row(s) in column '{col}'")
                                for _, row in matched.iterrows():
                                    row_dict = {
                                        k: v.title() if isinstance(v, str) else v
                                        for k, v in row.to_dict().items()
                                    }
                                    row_dict['Year'] = year
                                    match_rows.append(row_dict)

                    results_dict.extend(match_rows)

                except Exception as e:
                    print(f"❌ Error processing {file}: {e}")

    if results_dict:
        final_df = pd.DataFrame(results_dict)
        final_df.drop_duplicates(inplace=True)
        final_df.to_csv(output_path, index=False)
        print(f"\n✅ Extracted species saved to: {output_path}")
    else:
        print("\n⚠️ No species matched in any file. Nothing saved.")


In [ ]:
def extract_species_in_csv(input_dir, output_path, year_range=(2011, 2023), species_list=species_list):
    """
    Extract rows containing any species in species_list (case-insensitive, partial match)
    from all CSV files in input_dir for years within year_range.
    Matches can be in any column.
    """
    results_dict = []  # Store results in a list of dictionaries for later conversion to DataFrame

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                # Extract 4-digit year from filename
                match = re.search(r'\b(20\d{2})\b', file)
                if not match:
                    print(f"⚠️ Skipping {file} — No 4-digit year found in filename")
                    continue

                year = int(match.group())
                if year < year_range[0] or year > year_range[1]:
                    print(f"⚠️ Skipping {file} — Year {year} out of range {year_range}")
                    continue

                file_path = os.path.join(root, file)
                print(f"\n📄 Scanning file: {file} (Year: {year})")

                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')
                    df.dropna(how='all', inplace=True)
                    df.columns = df.columns.astype(str)

                    match_rows = []  # List to store rows that match any species

                    for species in species_list:
                        # Check for species matches in any column (case-insensitive)
                        for col in df.columns:
                            mask = df[col].astype(str).str.contains(species, case=False, na=False)
                            matched = df[mask]

                            if not matched.empty:
                                print(f"✅ Found '{species}' in {file} — {len(matched)} row(s) in column {col}")
                                # Add matched rows to the results list with the 'Year' added
                                for idx, row in matched.iterrows():
                                    match_rows.append(row.to_dict())  # Convert row to dict

                    # After processing the file, add the year information to each match
                    for row in match_rows:
                        row['Year'] = year

                    results_dict.extend(match_rows)  # Add the matched rows from this file

                except Exception as e:
                    print(f"❌ Error processing {file}: {e}")

    # Convert the accumulated results into a DataFrame
    if results_dict:
        final_df = pd.DataFrame(results_dict)
        final_df.drop_duplicates(inplace=True)  # Remove any duplicate rows
        final_df.to_csv(output_path, index=False)
        print(f"\n✅ Extracted species saved to: {output_path}")
    else:
        print("\n⚠️ No species matched in any file. Nothing saved.")


Check info of the datasets

In [ ]:
def check_info_of_the_datasets(input_dir):
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                print(f"📄 Processing: {file}")

                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

                    print({file})
                    df.info()
                    print('\n')

                except Exception as e:
                    print(f"❌ Error processing {file}: {e}")


Replace empty data with zeros and change data type

In [ ]:
def replace_empty_data_with_zeros(input_dir):
  for filename in os.listdir(input_dir):
    if filename.lower().endswith('.csv'):
      file_path = os.path.join(input_dir, filename)

      try:
        df = pd.read_csv(file_path)

        # Replace empty strings and NaNs with 0
        df.replace('', 0, inplace=True)
        df.fillna(0, inplace=True)

        # Convert columns [1:-1] (2nd to second-last) to float
        if df.shape[1] >= 3:
          cols_to_convert = df.columns[1:-1]
          df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)

        # Save the modified file back
        df.to_csv(file_path, index=False)
        print(f"✅ Replaced empty data with zeros in {filename}")

      except Exception as e:
        print(f"❌ Error processing {filename}: {e}")

  print("✅ Done")

Identify missing species and remove extra/unexpected species

In [ ]:
def check_and_remove_species_per_year(input_path, output_path, species_list_gear):
    """
    Validates and cleans species data per year:
    - Prints missing and extra species for each year.
    - Removes rows with species not in the expected list.
    - Saves cleaned CSV to the given output path.
    """
    # Load CSV
    df = pd.read_csv(input_path)

    # Normalize species column
    df['Species'] = df['Species'].astype(str).str.strip()

    # Group by year
    all_cleaned_rows = []

    for year, group in df.groupby('Year'):
        present_species = group['Species'].unique().tolist()

        # Find missing and extra species
        missing = [s for s in species_list_gear if s not in present_species]
        extra = [s for s in present_species if s not in species_list_gear]

        print(f"\n📅 Year {year}")
        if missing:
            print(f"⚠️ Missing species: {missing}")
        else:
            print("✅ All expected species found.")

        if extra:
            print(f"🗑️ Removing extra species: {extra}")

        # Keep only expected species
        filtered_group = group[group['Species'].isin(species_list_gear)].copy()
        all_cleaned_rows.append(filtered_group)

    # Concatenate and save
    cleaned_df = pd.concat(all_cleaned_rows, ignore_index=True)
    cleaned_df.to_csv(output_path, index=False)
    print(f"\n💾 Cleaned CSV saved to: {output_path}")


## Extract 4.5 related datasets

### Extract from csv in 4.5.2 Dataset - West Coast of Malaysia

In [ ]:
# Input & output paths
input_dir_1 = "/content/drive/My Drive/FYP_Fish/4.5.2 West"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_15.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_15.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_15.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_14.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_15.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_1, f'4.5.2 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")



 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1528
Total rows after merging: 1528
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1544
Total rows after merging: 1544
Saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
clean_csv_files(input_dir_1)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned 

In [ ]:
remove_unnecessary_columns_rows(input_dir_1, input_dir_1)

ℹ️ No rule for 2009 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2016.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned fi

In [ ]:
found_species_by_year = find_species_in_csv_files(input_dir_1, species_list)


🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv
⚠️ Skipping 4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv — Year 2009 out of range

🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
🔎 Processing 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — Year 2020
❌ Bilis/Bunga Air not found in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Pelata in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Selar in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found SelarKuning in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Kayu/Tongkol/AyaHitam in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Kayu/Tongkol/AyaKurik in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
❌ Kayu/Tongkol/Aya Selasih not found in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Kayu/Tongkol/AyaJalur in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Found Kayu/Tongkol/AyaPeluru in 4.5.2 - Jadual Penda

In [ ]:
summarize_species_matches(found_species_by_year)


📅 Year: 2011
✅ Total unique species found: 39

📅 Year: 2012
✅ Total unique species found: 39

📅 Year: 2013
✅ Total unique species found: 39

📅 Year: 2014
✅ Total unique species found: 39

📅 Year: 2015
✅ Total unique species found: 41

📅 Year: 2016
✅ Total unique species found: 49

📅 Year: 2017
✅ Total unique species found: 60

📅 Year: 2018
✅ Total unique species found: 60

📅 Year: 2019
✅ Total unique species found: 60

📅 Year: 2020
✅ Total unique species found: 61

📅 Year: 2021
✅ Total unique species found: 61

📅 Year: 2022
✅ Total unique species found: 57

📅 Year: 2023
✅ Total unique species found: 57


In [ ]:
drop_last_column(input_dir_1)

✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2016.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2010.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.2 - Jadual Pendaratan Ikan Laut 2012.csv — Empty rows/columns and last column removed


In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 14
🔢 Columns after: 13
✅ Last column dropped from 4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 14
🔢 Columns after: 13
✅ Last column dropped from 4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
add_header(input_dir_1, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /

In [ ]:
add_year_column(input_dir_1)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv with year: 2009",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv with year: 2020",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2022.csv with year: 2022",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2014.csv with year: 2014",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 - Jadual Pendaratan Ikan Laut 2021.csv with year: 2021",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.2 West/4.5.2 -

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.5.2 West/extracted_species - West.csv"
final_df = extract_species_in_csv(input_dir_1, output_path)

⚠️ Skipping 4.5.2 - Jadual Pendaratan Ikan Laut 2009.csv — Year 2009 out of range (2011, 2023)

📄 Scanning file: 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv (Year: 2020)
✅ Found 'Pelata' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Selar' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column Species
✅ Found 'SelarKuning' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaHitam' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaKurik' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaJalur' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaPeluru' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Found 'Selayang/Curut' in 4.5.2 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column Species
✅ Fo

### Extract from csv in 4.5.3 Dataset - East Coast of Malaysia

In [ ]:
# Input paths
input_dir_2 = "/content/drive/My Drive/FYP_Fish/4.5.3 East"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_17.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_18.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.5.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_17.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_18.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_17.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_18.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_17.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_18.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_17.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_18.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_2, f'4.5.3 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total rows before merging: 170
Total rows after merging: 170
Saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1528
Total rows after merging: 1528
Saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1552
Total rows after merging: 1552
Saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
clean_csv_files(input_dir_2)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned 

In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.5.2 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv
❌ File not found: /content/drive/My Drive/FYP_Fish/4.5.2 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows(input_dir_2, input_dir_2)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv
ℹ️ No rule for 2009 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned fi

In [ ]:
found_species_by_year = find_species_in_csv_files(input_dir_2, species_list)


🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
🔎 Processing 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — Year 2011
❌ Bilis/Bunga Air not found in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Pelata in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Selar in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found SelarKuning in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaHitam in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaKurik in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
❌ Kayu/Tongkol/Aya Selasih not found in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaJalur in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
❌ Kayu/Tongkol/AyaPeluru not found in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Selayang/Curut in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found BawalHitam in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found BawalPutih

In [ ]:
summarize_species_matches(found_species_by_year)


📅 Year: 2011
✅ Total unique species found: 39

📅 Year: 2012
✅ Total unique species found: 39

📅 Year: 2013
✅ Total unique species found: 39

📅 Year: 2014
✅ Total unique species found: 39

📅 Year: 2015
✅ Total unique species found: 41

📅 Year: 2016
✅ Total unique species found: 48

📅 Year: 2017
✅ Total unique species found: 59

📅 Year: 2018
✅ Total unique species found: 59

📅 Year: 2019
✅ Total unique species found: 59

📅 Year: 2020
✅ Total unique species found: 61

📅 Year: 2021
✅ Total unique species found: 61

📅 Year: 2022
✅ Total unique species found: 57

📅 Year: 2023
✅ Total unique species found: 57


In [ ]:
drop_last_column(input_dir_2)

✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2009.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2010.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.3 - Jadual Pendaratan Ikan Laut 2013.csv — Empty rows/columns and last column removed


In [ ]:
add_header(input_dir_2, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2009.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2022.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /

In [ ]:
add_year_column(input_dir_2)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv with year: 2011",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2009.csv with year: 2009",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2016.csv with year: 2016",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2010.csv with year: 2010",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.3 East/4.5.3 -

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.5.3 East/extracted_species - East.csv"
final_df = extract_species_in_csv(input_dir_2, output_path)


📄 Scanning file: 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv (Year: 2011)
✅ Found 'Pelata' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Selar' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 2 row(s) in column Species
✅ Found 'SelarKuning' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaHitam' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaKurik' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaJalur' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Selayang/Curut' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'BawalHitam' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'BawalPutih' in 4.5.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Bawa

### Extract from csv in 4.5.4 Dataset - Sarawak

In [ ]:
# Input paths
input_dir_3 = "/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_20.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_21.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_20.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_21.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_20.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_21.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_20.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_21.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_3, f'4.5.4 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 156
Total rows after merging: 156
Saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1528
Total rows after merging: 1528
Saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1544
Total rows after merging: 1544
Saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
clean_csv_files(input_dir_3)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ik

In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows(input_dir_3, input_dir_3)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Saved cleaned file to /content/

In [ ]:
found_species_by_year = find_species_in_csv_files(input_dir_3, species_list)


🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
🔎 Processing 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — Year 2011
❌ Bilis/Bunga Air not found in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Pelata in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Selar in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found SelarKuning in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaHitam in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaKurik in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
❌ Kayu/Tongkol/Aya Selasih not found in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Kayu/Tongkol/AyaJalur in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
❌ Kayu/Tongkol/AyaPeluru not found in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found Selayang/Curut in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found BawalHitam in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Found BawalPu

In [ ]:
summarize_species_matches(found_species_by_year)


📅 Year: 2011
✅ Total unique species found: 39

📅 Year: 2012
✅ Total unique species found: 39

📅 Year: 2013
✅ Total unique species found: 39

📅 Year: 2014
✅ Total unique species found: 39

📅 Year: 2015
✅ Total unique species found: 41

📅 Year: 2016
✅ Total unique species found: 48

📅 Year: 2017
✅ Total unique species found: 59

📅 Year: 2018
✅ Total unique species found: 59

📅 Year: 2019
✅ Total unique species found: 59

📅 Year: 2020
✅ Total unique species found: 61

📅 Year: 2021
✅ Total unique species found: 61

📅 Year: 2022
✅ Total unique species found: 57

📅 Year: 2023
✅ Total unique species found: 57


In [ ]:
drop_last_column(input_dir_3)

✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2010.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2014.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.4 - Jadual Pendaratan Ikan Laut 2013.csv — Empty rows/columns and last column removed


In [ ]:
add_header(input_dir_3, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 201

In [ ]:
add_year_column(input_dir_3)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv with year: 2011",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2022.csv with year: 2022",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2010.csv with year: 2010",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2020.csv with year: 2020",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/4.5.4 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fi

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.5.4 Sarawak/extracted_species - Sarawak.csv"
final_df = extract_species_in_csv(input_dir_3, output_path)


📄 Scanning file: 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv (Year: 2011)
✅ Found 'Pelata' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Selar' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 2 row(s) in column Species
✅ Found 'SelarKuning' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaHitam' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaKurik' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaJalur' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Selayang/Curut' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'BawalHitam' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'BawalPutih' in 4.5.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column Species
✅ Found 'Bawa

### Extract from csv in 4.5.5 Dataset - Sabah

In [ ]:
# Input paths
input_dir_4 = "/content/drive/My Drive/FYP_Fish/4.5.5 Sabah"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_23.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_24.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_23.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_24.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_23.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_24.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_23.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_24.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_4, f'4.5.5 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1458
Total rows after merging: 1458
Saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1536
Total rows after merging: 1536
Saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
clean_csv_files(input_dir_4)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2018.csv
✅

In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows(input_dir_4, input_dir_4)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv
ℹ️ No rule for 2008 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Saved cleaned file to /content/drive/My Drive/F

In [ ]:
found_species_by_year = find_species_in_csv_files(input_dir_4, species_list)


🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
🔎 Processing 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — Year 2013
❌ Bilis/Bunga Air not found in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Pelata in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Selar in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found SelarKuning in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Kayu/Tongkol/AyaHitam in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Kayu/Tongkol/AyaKurik in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
❌ Kayu/Tongkol/Aya Selasih not found in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Kayu/Tongkol/AyaJalur in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
❌ Kayu/Tongkol/AyaPeluru not found in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found Selayang/Curut in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found BawalHitam in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Found BawalPuti

In [ ]:
summarize_species_matches(found_species_by_year)


📅 Year: 2011
✅ Total unique species found: 39

📅 Year: 2012
✅ Total unique species found: 39

📅 Year: 2013
✅ Total unique species found: 39

📅 Year: 2014
✅ Total unique species found: 39

📅 Year: 2015
✅ Total unique species found: 40

📅 Year: 2016
✅ Total unique species found: 46

📅 Year: 2017
✅ Total unique species found: 58

📅 Year: 2018
✅ Total unique species found: 58

📅 Year: 2019
✅ Total unique species found: 58

📅 Year: 2020
✅ Total unique species found: 61

📅 Year: 2021
✅ Total unique species found: 61

📅 Year: 2022
✅ Total unique species found: 57

📅 Year: 2023
✅ Total unique species found: 57


In [ ]:
drop_last_column(input_dir_4)

✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2008.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2018.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.5 - Jadual Pendaratan Ikan Laut 2011.csv — Empty rows/columns and last column removed


In [ ]:
add_header(input_dir_4, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2008.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2018.csv',
 'Added he

In [ ]:
add_year_column(input_dir_4)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2017.csv with year: 2017",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2022.csv with year: 2022",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2020.csv with year: 2020",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2008.csv with year: 2008",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah/4.5.5 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.5 Sabah

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.5.5 Sabah/extracted_species - Sabah.csv"
final_df = extract_species_in_csv(input_dir_4,output_path)


📄 Scanning file: 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv (Year: 2013)
✅ Found 'Pelata' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Selar' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 2 row(s) in column Species
✅ Found 'SelarKuning' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaHitam' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaKurik' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaJalur' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Selayang/Curut' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'BawalHitam' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'BawalPutih' in 4.5.5 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column Species
✅ Found 'Bawa

### Extract from csv in 4.5.6 Dataset - Labuan

In [ ]:
# Input paths
input_dir_5 = "/content/drive/My Drive/FYP_Fish/4.5.6 Labuan"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.5.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_27.csv')
     ],
     "2020": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual (Table) 4.5.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2020', 'Jadual_Page_27.csv')
     ],
     "2022": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual (Table) 4.5.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2022', 'Jadual_Page_27.csv')
    ],
    "2023": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual (Table) 4.5.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_26.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2023', 'Jadual_Page_27.csv')
    ]
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_5, f'4.5.6 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2020
Total rows before merging: 1122
Total rows after merging: 1122
Saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv

 Year: 2022
Total rows before merging: 1337
Total rows after merging: 1337
Saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2022.csv

 Year: 2023
Total rows before merging: 1536
Total rows after merging: 1536
Saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
clean_csv_files(input_dir_5)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2

In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.5.6 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.6 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 15
🔢 Columns after: 14
✅ Last column dropped from 4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows(input_dir_5, input_dir_5)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Saved cleaned file to /content/drive/My

In [ ]:
found_species_by_year = find_species_in_csv_files(input_dir_5, species_list)


🔍 Processing /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
🔎 Processing 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — Year 2018
❌ Bilis/Bunga Air not found in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Pelata in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Selar in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found SelarKuning in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Kayu/Tongkol/AyaHitam in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Kayu/Tongkol/AyaKurik in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
❌ Kayu/Tongkol/Aya Selasih not found in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Kayu/Tongkol/AyaJalur in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
❌ Kayu/Tongkol/AyaPeluru not found in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found Selayang/Curut in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found BawalHitam in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Found BawalPut

In [ ]:
summarize_species_matches(found_species_by_year)


📅 Year: 2011
✅ Total unique species found: 39

📅 Year: 2012
✅ Total unique species found: 39

📅 Year: 2013
✅ Total unique species found: 39

📅 Year: 2014
✅ Total unique species found: 39

📅 Year: 2015
✅ Total unique species found: 41

📅 Year: 2016
✅ Total unique species found: 47

📅 Year: 2017
✅ Total unique species found: 59

📅 Year: 2018
✅ Total unique species found: 59

📅 Year: 2019
✅ Total unique species found: 59

📅 Year: 2020
✅ Total unique species found: 61

📅 Year: 2021
✅ Total unique species found: 61

📅 Year: 2022
✅ Total unique species found: 57

📅 Year: 2023
✅ Total unique species found: 57


In [ ]:
drop_last_column(input_dir_5)

✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2010.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2016.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2014.csv — Empty rows/columns and last column removed
✅ Cleaned 4.5.6 - Jadual Pendaratan Ikan Laut 2022.csv — Empty rows/columns and last column removed


In [ ]:
add_header(input_dir_5, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2014.csv',
 

In [ ]:
add_year_column(input_dir_5)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv with year: 2018",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2020.csv with year: 2020",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.6 Labuan/4.5.6 - Jadual Pendaratan Ikan Laut 2010.csv with year: 2010",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.5.

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.5.6 Labuan/extracted_species - Labuan.csv"
final_df = extract_species_in_csv(input_dir_5, output_path)


📄 Scanning file: 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv (Year: 2018)
✅ Found 'Pelata' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Selar' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column Species
✅ Found 'SelarKuning' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaHitam' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaKurik' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Kayu/Tongkol/AyaJalur' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Selayang/Curut' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'BawalHitam' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'BawalPutih' in 4.5.6 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column Species
✅ Found 'Bawa

## Extracted 4.6 related dataset

### Extract from csv in 4.6.2 Dataset - West Coast of Malaysia

In [ ]:
# Input paths
input_dir_12 = "/content/drive/My Drive/FYP_Fish/4.6.2 West"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.6.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_35.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_36.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.6.2.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_35.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_36.csv')
     ],
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_12, f'4.6.2 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total rows before merging: 171
Total rows after merging: 171
Saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
clean_csv_files_gear(input_dir_12)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned 

<ipython-input-672-1635999794>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0         449.0
1        1678.0
2        9044.0
3       17904.0
4        1244.0
         ...   
213     13.9375
214      3.8078
215         0.0
216    660.1341
217    918.6192
Name: Trawl Nets, Length: 218, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)


In [ ]:
check_info_of_the_datasets(input_dir_12)

📄 Processing: 4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
{'4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       3 non-null      object
 1   1       3 non-null      object
 2   2       3 non-null      object
 3   3       3 non-null      object
 4   4       3 non-null      object
 5   5       3 non-null      object
dtypes: object(6)
memory usage: 276.0+ bytes


📄 Processing: 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv
{'4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   0       4 non-null      object
 1   1       101 non-null    object
 2   2       99 non-null     object
 3   3       99 non-null     object
 4   4       98 non-null    

In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv',
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows_fishing_gear(input_dir_12, input_dir_12)

ℹ️ No rule for 2007 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv
ℹ️ No rule for 2008 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv
ℹ️ No rule for 2009 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendarata

In [ ]:
add_header(input_dir_12, gear_header)

['Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2008.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2009.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /

In [ ]:
add_year_column(input_dir_12)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv with year: 2007",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv with year: 2021",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv with year: 2014",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2008.csv with year: 2008",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.2 West/4.6.2 -

In [ ]:
check_info_of_the_datasets(input_dir_12)

📄 Processing: 4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
{'4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               4 non-null      object 
 1   Trawl Nets            4 non-null      object 
 2   Fish Purse Seines     4 non-null      object 
 3   Anchovy Purse Seines  4 non-null      object 
 4   Other Seines          4 non-null      object 
 5   Drift/Gill Nets       4 non-null      object 
 6   Lift Nets             4 non-null      int64  
 7   Stationary Traps      0 non-null      float64
 8   Portable Traps        0 non-null      float64
 9   Hooks & Lines         0 non-null      float64
 10  Bag Nets              0 non-null      float64
 11  Barrier Nets          0 non-null      float64
 12  Push/Scoop Nets       0 non-null      float64
 13  ShellfishCollection  

In [ ]:
replace_empty_data_with_zeros(input_dir_12)

✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Replaced empty data with zeros in 4.6.2 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Replaced empty data with z

In [ ]:
check_info_of_the_datasets(input_dir_12)

📄 Processing: 4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv
{'4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               4 non-null      object 
 1   Trawl Nets            4 non-null      float64
 2   Fish Purse Seines     4 non-null      float64
 3   Anchovy Purse Seines  4 non-null      float64
 4   Other Seines          4 non-null      float64
 5   Drift/Gill Nets       4 non-null      float64
 6   Lift Nets             4 non-null      float64
 7   Stationary Traps      4 non-null      float64
 8   Portable Traps        4 non-null      float64
 9   Hooks & Lines         4 non-null      float64
 10  Bag Nets              4 non-null      float64
 11  Barrier Nets          4 non-null      float64
 12  Push/Scoop Nets       4 non-null      float64
 13  ShellfishCollection  

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.6.2 West/extracted_species - West.csv"
final_df = extract_gear_in_csv(input_dir_12, output_path, species_list=species_list_gear)

⚠️ Skipping 4.6.2 - Jadual Pendaratan Ikan Laut 2007.csv — Year 2007 out of range (2011, 2023)

📄 Scanning file: 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv (Year: 2012)
✅ Found 'Kerisibali' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.2 - Jadual Pendaratan Ikan Laut 2012.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.2 - Jadual Penda

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Selar' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.2 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in co

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in c

...


📄 Scanning file: 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv (Year: 2022)
✅ Found 'Kerisibali' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongk

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)



📄 Scanning file: 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv (Year: 2023)
✅ Found 'Kerisibali' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongk

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.2 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in c

In [ ]:
input_path = "/content/drive/My Drive/FYP_Fish/4.6.2 West/extracted_species - West.csv"
output_path = "/content/drive/My Drive/FYP_Fish/fishing_gear_files/cleaned_species_west.csv"

check_and_remove_species_per_year(input_path, output_path, species_list_gear)


📅 Year 2011
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2012
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2013
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2014
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2015
✅ All expected species found.
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayakurik', 'Kayu/Tongkol/Ayaselasih', 'Kayu/Tongkol/Ayajalur']

📅 Year 2017
✅ All expected species 

### Extract from csv in 4.6.3 Dataset - East Coast of Malaysia

In [ ]:
# Input paths
input_dir_13 = "/content/drive/My Drive/FYP_Fish/4.6.3 East"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.6.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_38.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_39.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.6.3.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_38.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_39.csv')
     ],
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    column_counts = [len(df.columns) for df in dfs]
    total_columns_before = sum(column_counts)
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)
    total_columns_after = len(merged_df.columns)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_13, f'4.6.3 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total columns before merging: {total_columns_before}")
    print(f"Total columns after merging: {total_columns_after}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total columns before merging: 42
Total columns after merging: 14
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total columns before merging: 54
Total columns after merging: 18
Total rows before merging: 170
Total rows after merging: 170
Saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
clean_csv_files_gear(input_dir_13)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned 

<ipython-input-672-1635999794>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0      1361.0
1       410.0
2      3143.0
3      6372.0
4       204.0
        ...  
213      24.0
214      70.0
215       0.0
216    1434.0
217       0.0
Name: Trawl Nets, Length: 218, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)


In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.6.3 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.3 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows_fishing_gear(input_dir_13, input_dir_13)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv
ℹ️ No rule for 2006 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2013.csv
ℹ️ No rule for 2008 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendarata

In [ ]:
add_header(input_dir_13, gear_header)

['Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /

In [ ]:
add_year_column(input_dir_13)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv with year: 2016",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv with year: 2006",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv with year: 2014",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.3 East/4.6.3 -

In [ ]:
check_info_of_the_datasets(input_dir_13)

📄 Processing: 4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv
{'4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv'}
<class 'pandas.core.frame.DataFrame'>
Index: 168 entries, Kebasi to Jumlah(Total)
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Species               164 non-null    object
 1   Trawl Nets            165 non-null    object
 2   Fish Purse Seines     165 non-null    object
 3   Anchovy Purse Seines  166 non-null    object
 4   Other Seines          163 non-null    object
 5   Drift/Gill Nets       165 non-null    object
 6   Lift Nets             166 non-null    object
 7   Stationary Traps      162 non-null    object
 8   Portable Traps        166 non-null    object
 9   Hooks & Lines         166 non-null    object
 10  Bag Nets              166 non-null    object
 11  Barrier Nets          166 non-null    object
 12  Push/Scoop Nets       166 non-null    object
 13  ShellfishCollection   

In [ ]:
replace_empty_data_with_zeros(input_dir_13)

✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Replaced empty data with zeros in 4.6.3 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Replaced empty data with z

In [ ]:
check_info_of_the_datasets(input_dir_13)

📄 Processing: 4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv
{'4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               168 non-null    object 
 1   Trawl Nets            168 non-null    float64
 2   Fish Purse Seines     168 non-null    float64
 3   Anchovy Purse Seines  168 non-null    float64
 4   Other Seines          168 non-null    float64
 5   Drift/Gill Nets       168 non-null    float64
 6   Lift Nets             168 non-null    float64
 7   Stationary Traps      168 non-null    float64
 8   Portable Traps        168 non-null    float64
 9   Hooks & Lines         168 non-null    float64
 10  Bag Nets              168 non-null    float64
 11  Barrier Nets          168 non-null    float64
 12  Push/Scoop Nets       168 non-null    float64
 13  ShellfishCollecti

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.6.3 East/extracted_species - East.csv"
final_df = extract_gear_in_csv(input_dir_13, output_path, species_list=species_list_gear)


📄 Scanning file: 4.6.3 - Jadual Pendaratan Ikan Laut 2016.csv (Year: 2016)
⚠️ Skipping 4.6.3 - Jadual Pendaratan Ikan Laut 2006.csv — Year 2006 out of range (2011, 2023)

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)




📄 Scanning file: 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv (Year: 2012)
✅ Found 'Kerisibali' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.3 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.3 - Jadual Penda

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.3 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.3 - Jadual Pendaratan Ikan Laut 2015.csv (Year: 2015)
✅ Found 'Kerisibali' in 4.6.3 - Jadual 

...

✅ Found 'Kerisibali' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.3 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv (Year: 2020)


<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.3 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in c

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.3 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in c

In [ ]:
input_path = "/content/drive/My Drive/FYP_Fish/4.6.3 East/extracted_species - East.csv"
output_path = "/content/drive/My Drive/FYP_Fish/fishing_gear_files/cleaned_species_east.csv"

check_and_remove_species_per_year(input_path, output_path, species_list_gear)


📅 Year 2011
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2012
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2013
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2014
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2015
✅ All expected species found.
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayakurik', 'Kayu/Tongkol/Ayaselasih', 'Kayu/Tongkol/Ayajalur']

📅 Year 2017
✅ All expected species 

### Extract from csv in 4.6.4 Dataset - Sarawak

In [ ]:
# Input paths
input_dir_14 = "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.6.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_41.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_42.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.6.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_41.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_42.csv')
     ],
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_14, f'4.6.4 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total rows before merging: 170
Total rows after merging: 170
Saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
clean_csv_files_gear(input_dir_14)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ik

<ipython-input-672-1635999794>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0       597.0
1       186.0
2       928.0
3        31.0
4       545.0
        ...  
213    1468.0
214       0.0
215      49.0
216    1149.0
217      33.0
Name: Trawl Nets, Length: 218, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)


In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.6.4 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.4 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows_fishing_gear(input_dir_14, input_dir_14)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv
ℹ️ No rule for 2008 — keeping original
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 202

In [ ]:
add_header(input_dir_14, gear_header)

['Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2008.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 202

In [ ]:
add_year_column(input_dir_14)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv with year: 2017",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv with year: 2011",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2008.csv with year: 2008",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2010.csv with year: 2010",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fi

In [ ]:
check_info_of_the_datasets(input_dir_14)

📄 Processing: 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv
{'4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Species               182 non-null    object
 1   Trawl Nets            181 non-null    object
 2   Fish Purse Seines     181 non-null    object
 3   Anchovy Purse Seines  181 non-null    object
 4   Other Seines          181 non-null    object
 5   Drift/Gill Nets       178 non-null    object
 6   Lift Nets             181 non-null    object
 7   Stationary Traps      181 non-null    object
 8   Portable Traps        179 non-null    object
 9   Hooks & Lines         181 non-null    object
 10  Bag Nets              181 non-null    object
 11  Barrier Nets          181 non-null    object
 12  Push/Scoop Nets       181 non-null    object
 13  ShellfishCollection   181 non-nu

In [ ]:
replace_empty_data_with_zeros(input_dir_14)

✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2006.csv
✅ Replaced empty data with zeros in 4.6.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Replaced empty data with z

In [ ]:
check_info_of_the_datasets(input_dir_14)

📄 Processing: 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv
{'4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183 entries, 0 to 182
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               183 non-null    object 
 1   Trawl Nets            183 non-null    float64
 2   Fish Purse Seines     183 non-null    float64
 3   Anchovy Purse Seines  183 non-null    float64
 4   Other Seines          183 non-null    float64
 5   Drift/Gill Nets       183 non-null    float64
 6   Lift Nets             183 non-null    float64
 7   Stationary Traps      183 non-null    float64
 8   Portable Traps        183 non-null    float64
 9   Hooks & Lines         183 non-null    float64
 10  Bag Nets              183 non-null    float64
 11  Barrier Nets          183 non-null    float64
 12  Push/Scoop Nets       183 non-null    float64
 13  ShellfishCollecti

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/extracted_species - Sarawak.csv"
final_df = extract_gear_in_csv(input_dir_14, output_path, species_list=species_list_gear)


📄 Scanning file: 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv (Year: 2017)
✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongk

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Tenggiri' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 6 row(s) in column 'Species'
✅ Found 'Selarkuning' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.4 - Jadual Pendaratan Ikan Laut 2017.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv (Year: 2011)
✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2011.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendar

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in c

...

✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in c

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.4 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in c

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.4 - Jadual Pendaratan Ikan Laut 2014.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.4 - Jadual Pendaratan Ikan Laut 2013.csv (Year: 2013)
✅ Found 'Kerisibali' in 4.6.4 - Jadual 

In [ ]:
input_path = "/content/drive/My Drive/FYP_Fish/4.6.4 Sarawak/extracted_species - Sarawak.csv"
output_path = "/content/drive/My Drive/FYP_Fish/fishing_gear_files/cleaned_species_sarawak.csv"

check_and_remove_species_per_year(input_path, output_path, species_list_gear)


📅 Year 2011
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2012
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2013
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2014
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2015
✅ All expected species found.
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayakurik', 'Kayu/Tongkol/Ayaselasih', 'Kayu/Tongkol/Ayajalur']

📅 Year 2017
✅ All expected species 

### Extract from csv in 4.6.5 Dataset - Sabah

In [ ]:
# Input paths
input_dir_15 = "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.6.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_44.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_45.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.6.5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_44.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_45.csv')
     ],
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_15, f'4.6.5 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total rows before merging: 171
Total rows after merging: 171
Saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
clean_csv_files_gear(input_dir_15)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2006.csv
✅

<ipython-input-672-1635999794>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0       202.0126
1       958.9929
2         777.31
3       821.3154
4      1200.6219
         ...    
213          0.0
214          0.0
215          0.0
216          0.0
217          0.0
Name: Trawl Nets, Length: 218, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)


In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows_fishing_gear(input_dir_15, input_dir_15)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv
ℹ️ No rule for 2010 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2014.csv
ℹ️ No rule for 2006 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2016.csv
ℹ️ No rule

In [ ]:
add_header(input_dir_15, gear_header)

['Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2017.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2014.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2006.csv',
 'Added he

In [ ]:
add_year_column(input_dir_15)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv with year: 2018",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv with year: 2022",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv with year: 2010",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah/4.6.5 - Jadual Pendaratan Ikan Laut 2017.csv with year: 2017",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.5 Sabah

In [ ]:
check_info_of_the_datasets(input_dir_15)

📄 Processing: 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
{'4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               180 non-null    object 
 1   Trawl Nets            180 non-null    float64
 2   Fish Purse Seines     178 non-null    float64
 3   Anchovy Purse Seines  180 non-null    float64
 4   Other Seines          178 non-null    float64
 5   Drift/Gill Nets       175 non-null    float64
 6   Lift Nets             179 non-null    float64
 7   Stationary Traps      180 non-null    float64
 8   Portable Traps        178 non-null    float64
 9   Hooks & Lines         180 non-null    float64
 10  Bag Nets              180 non-null    float64
 11  Barrier Nets          180 non-null    float64
 12  Push/Scoop Nets       180 non-null    float64
 13  ShellfishCollecti

In [ ]:
replace_empty_data_with_zeros(input_dir_15)

✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2006.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Replaced empty data with zeros in 4.6.5 - Jadual Pendaratan Ikan Laut 2021.csv
✅ Replaced empty data with z

In [ ]:
check_info_of_the_datasets(input_dir_15)

📄 Processing: 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv
{'4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               180 non-null    object 
 1   Trawl Nets            180 non-null    float64
 2   Fish Purse Seines     180 non-null    float64
 3   Anchovy Purse Seines  180 non-null    float64
 4   Other Seines          180 non-null    float64
 5   Drift/Gill Nets       180 non-null    float64
 6   Lift Nets             180 non-null    float64
 7   Stationary Traps      180 non-null    float64
 8   Portable Traps        180 non-null    float64
 9   Hooks & Lines         180 non-null    float64
 10  Bag Nets              180 non-null    float64
 11  Barrier Nets          180 non-null    float64
 12  Push/Scoop Nets       180 non-null    float64
 13  ShellfishCollecti

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/extracted_species - Sabah.csv"
final_df = extract_gear_in_csv(input_dir_15, output_path, species_list=species_list_gear)


📄 Scanning file: 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv (Year: 2018)
✅ Found 'Kerisibali' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongk

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Bawalputih' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 6 row(s) in column 'Species'
✅ Found 'Selarkuning' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.5 - Jadual Pendaratan Ikan Laut 2018.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv (Year: 2022)
✅ Found 'Kerisibali' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.5 - Ja

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 4 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 6 row(s) in column 'Species'
✅ Found 'Selarkuning' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.5 - Jadual Pendaratan Ikan Laut 2022.csv — 1 row(s) in column 'Species'
⚠️ Skipping 4.6.5 - Jadual Pendaratan Ikan Laut 2010.csv — Year 2010 out of range (2011, 2023)

📄 Scanning file: 4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv (Year: 2015)
✅ Found 'Kerisibali' in 4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2015.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadua

...

✅ Found 'Yu' in 4.6.5 - Jadual Pendaratan Ikan Laut 2021.csv — 7 row(s) in column 'Species'
✅ Found 'Selarkuning' in 4.6.5 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.5 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv (Year: 2020)
✅ Found 'Kerisibali' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.5 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.5 - Jadua

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Kerisibali' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in c

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Ketamlaut' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Selar' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.5 - Jadual Pendaratan Ikan Laut 2023.csv — 4 row(s) in col

In [ ]:
input_path = "/content/drive/My Drive/FYP_Fish/4.6.5 Sabah/extracted_species - Sabah.csv"
output_path = "/content/drive/My Drive/FYP_Fish/fishing_gear_files/cleaned_species_sabah.csv"

check_and_remove_species_per_year(input_path, output_path, species_list_gear)


📅 Year 2011
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2012
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2013
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2014
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2015
✅ All expected species found.
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayakurik', 'Kayu/Tongkol/Ayaselasih', 'Kayu/Tongkol/Ayajalur']

📅 Year 2017
✅ All expected species 

### Extract from csv in 4.6.6 Dataset - Labuan

In [ ]:
# Input paths
input_dir_16 = "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan"

In [ ]:
file_paths = {
     "2015": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual(Table) 4.6.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_47.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2015', 'Jadual_Page_48.csv')
     ],
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.6.6.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_47.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_48.csv')
     ],
}

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_16, f'4.6.6 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\n Year: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Saved: {output_file}")


 Year: 2015
Total rows before merging: 155
Total rows after merging: 155
Saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv

 Year: 2016
Total rows before merging: 170
Total rows after merging: 170
Saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
clean_csv_files_gear(input_dir_16)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2

<ipython-input-672-1635999794>:22: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0         154.0
1         182.0
2         259.0
3         594.0
4         204.0
         ...   
192         0.0
193         0.0
194         0.0
195    103.7898
196     97.5418
Name: Trawl Nets, Length: 197, dtype: object' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.iloc[:, 1] = df.iloc[:, 1].astype(str).str.replace(' ', '', regex=False)


In [ ]:
unclean = [
    '/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2022.csv',
    '/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv'
]

for file_path in unclean:
    if not os.path.isfile(file_path):
        print(f"❌ File not found: {file_path}")
        continue

    try:
        # Read the CSV file
        df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')

        # Show the number of columns before dropping
        print(f"\n📄 Processing {os.path.basename(file_path)}")
        print(f"🔢 Columns before: {len(df.columns)}")

        # Drop the last column
        df.drop(df.columns[-1], axis=1, inplace=True)

        # Show the number of columns after dropping
        print(f"🔢 Columns after: {len(df.columns)}")

        # Save the cleaned file (overwrite original)
        df.to_csv(file_path, index=False)
        print(f"✅ Last column dropped from {os.path.basename(file_path)}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")



📄 Processing 4.6.6 - Jadual Pendaratan Ikan Laut 2022.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.6 - Jadual Pendaratan Ikan Laut 2022.csv

📄 Processing 4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv
🔢 Columns before: 17
🔢 Columns after: 16
✅ Last column dropped from 4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv


In [ ]:
remove_unnecessary_columns_rows_fishing_gear(input_dir_16, input_dir_16)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv
ℹ️ No rule for 2009 — keeping original
ℹ️ No rule for 2008 — keeping original
ℹ️ No rule for 2006 — keeping original
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - 

In [ ]:
add_header(input_dir_16, gear_header)

['Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2009.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2008.csv',
 

In [ ]:
add_year_column(input_dir_16)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv with year: 2016",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv with year: 2023",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv with year: 2011",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.6 Labuan/4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv with year: 2014",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.6.

In [ ]:
check_info_of_the_datasets(input_dir_16)

📄 Processing: 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
{'4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Species               99 non-null     object
 1   Trawl Nets            98 non-null     object
 2   Fish Purse Seines     98 non-null     object
 3   Anchovy Purse Seines  98 non-null     object
 4   Other Seines          98 non-null     object
 5   Drift/Gill Nets       98 non-null     object
 6   Lift Nets             98 non-null     object
 7   Stationary Traps      97 non-null     object
 8   Portable Traps        98 non-null     object
 9   Hooks & Lines         98 non-null     object
 10  Bag Nets              98 non-null     object
 11  Barrier Nets          98 non-null     object
 12  Push/Scoop Nets       98 non-null     object
 13  ShellfishCollection   98 non-null 

In [ ]:
replace_empty_data_with_zeros(input_dir_16)

✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2023.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2014.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2009.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2008.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2006.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2022.csv
✅ Replaced empty data with zeros in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Replaced empty data with z

In [ ]:
check_info_of_the_datasets(input_dir_16)

📄 Processing: 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv
{'4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv'}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               99 non-null     object 
 1   Trawl Nets            99 non-null     float64
 2   Fish Purse Seines     99 non-null     float64
 3   Anchovy Purse Seines  99 non-null     float64
 4   Other Seines          99 non-null     float64
 5   Drift/Gill Nets       99 non-null     float64
 6   Lift Nets             99 non-null     float64
 7   Stationary Traps      99 non-null     float64
 8   Portable Traps        99 non-null     float64
 9   Hooks & Lines         99 non-null     float64
 10  Bag Nets              99 non-null     float64
 11  Barrier Nets          99 non-null     float64
 12  Push/Scoop Nets       99 non-null     float64
 13  ShellfishCollection

In [ ]:
output_path="/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/extracted_species - Labuan.csv"
final_df = extract_gear_in_csv(input_dir_16, output_path, species_list=species_list_gear)


📄 Scanning file: 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv (Year: 2012)
✅ Found 'Kerisibali' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'


<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Tenggiri' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.6 - Jadual Pendaratan Ikan Laut 2012.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv (Year: 2013)
✅ Found 'Kerisibali' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'
✅ Found 'Ketamlaut' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'


<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Tenggiri' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 3 row(s) in column 'Species'
✅ Found 'Siakap' in 4.6.6 - Jadual Pendaratan Ikan Laut 2013.csv — 1 row(s) in column 'Species'

📄 Scanning file: 4.6.6 - Jadual Pendaratan Ikan Laut 2016.csv (Year: 2016)


...

✅ Found 'Selar' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.6 - Jadual Pendaratan Ikan Laut 2020.csv — 1 row(s) in co

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Selar' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.6 - Jadual Pendaratan Ikan Laut 2019.csv — 1 row(s) in co

<ipython-input-682-1457590799>:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_lower = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


✅ Found 'Alu-Alu/Kacang-Kacang' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bijinangka' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 3 row(s) in column 'Species'
✅ Found 'Kerisi' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 2 row(s) in column 'Species'
✅ Found 'Sebelah' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Gelama/Tengkerong' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalhitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Kayu/Tongkol/Ayahitam' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Tenggiri' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Bawalputih' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 1 row(s) in column 'Species'
✅ Found 'Yu' in 4.6.6 - Jadual Pendaratan Ikan Laut 2021.csv — 7 row(s) in colum

In [ ]:
input_path = "/content/drive/My Drive/FYP_Fish/4.6.6 Labuan/extracted_species - Labuan.csv"
output_path = "/content/drive/My Drive/FYP_Fish/fishing_gear_files/cleaned_species_labuan.csv"

check_and_remove_species_per_year(input_path, output_path, species_list_gear)


📅 Year 2011
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2012
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2013
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2014
⚠️ Missing species: ['Selar', 'Alu-Alu/Kacang-Kacang', 'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Bawalputih', 'Selarkuning']
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayajalur']

📅 Year 2015
✅ All expected species found.
🗑️ Removing extra species: ['Puyulaut', 'Kayu/Tongkol/Ayakurik', 'Kayu/Tongkol/Ayaselasih', 'Kayu/Tongkol/Ayajalur']

📅 Year 2017
✅ All expected species 

# Handle Extracted Species Dataset

In [ ]:
# Base directory
input_base_location = "/content/drive/My Drive/FYP_Fish/4.5 location"

In [ ]:
# add location column
def add_location_column(input_dir):
    """
    Add 'location' column to all CSV files in input_dir based on filename.
    Save each file back to its original location.
    """
    for file in os.listdir(input_dir):
        if file.lower().endswith(".csv"):
            file_path = os.path.join(input_dir, file)
            print(f"Processing file: {file}")

            # Infer location from filename
            filename_lower = file.lower()
            print(filename_lower)
            if "east" in filename_lower:
                location = "East Peninsular Malaysia"
            elif "west" in filename_lower:
                location = "West Peninsular Malaysia"
            elif "sabah" in filename_lower:
                location = "Sabah"
            elif "sarawak" in filename_lower:
                location = "Sarawak"
            elif "labuan" in filename_lower:
                location = "Labuan"
            else:
                location = "Unknown"

            try:
                df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')
                df.dropna(how='all', inplace=True)

                if df.empty:
                    print(f"Skipping {file} — DataFrame is empty")
                    continue

                df["location"] = location
                df.to_csv(file_path, index=False)
                print(f"Updated with location: {location}\n")

            except Exception as e:
                print(f"Error processing {file}: {e}")

In [ ]:
add_location_column(input_base_location)

Processing file: extracted_species - Sabah.csv
extracted_species - sabah.csv
Updated with location: Sabah

Processing file: extracted_species - East.csv
extracted_species - east.csv
Updated with location: East Peninsular Malaysia

Processing file: extracted_species - Sarawak.csv
extracted_species - sarawak.csv
Updated with location: Sarawak

Processing file: extracted_species - West.csv
extracted_species - west.csv
Updated with location: West Peninsular Malaysia

Processing file: extracted_species - Labuan.csv
extracted_species - labuan.csv
Updated with location: Labuan



In [ ]:
# combine all csv into a df
def combine_csv_to_df(directory):
    """Combines all CSV files in a directory into a single pandas DataFrame.

    Args:
        directory: The path to the directory containing the CSV files.

    Returns:
        A pandas DataFrame containing the combined data, or None if an error occurs.
    """
    all_files = glob.glob(os.path.join(directory, "*.csv"))

    if not all_files:
        print(f"No CSV files found in {directory}")
        return None

    df_list = []
    for filename in all_files:
        try:
            df = pd.read_csv(filename, index_col=None, header=0)
            df_list.append(df)
        except pd.errors.EmptyDataError:
            print(f"Warning: Skipping empty file {filename}")
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            return None

    if not df_list:
        print("No valid CSV files found to combine")
        return None

    combined_df = pd.concat(df_list, axis=0, ignore_index=True)
    return combined_df

In [ ]:
combined_data = combine_csv_to_df(input_base_location)

In [ ]:
combined_data.head()

,Species,January,February,March,April,May,June,July,August,September,October,November,December,Year,location
0,Pelata,0.0,1.0,1.0,1.0,1.0,4.0,3.0,4.0,2.0,2.0,1.0,4.0,2013,Sabah
1,Selar,302.0,319.0,483.0,689.0,842.0,805.0,978.0,1188.0,942.0,734.0,397.0,312.0,2013,Sabah
2,SelarKuning,185.0,363.0,108.0,326.0,320.0,450.0,381.0,247.0,250.0,157.0,172.0,173.0,2013,Sabah
3,Kayu/Tongkol/AyaHitam,426.0,750.0,357.0,546.0,615.0,623.0,488.0,536.0,501.0,455.0,323.0,194.0,2013,Sabah
4,Kayu/Tongkol/AyaKurik,31.0,26.0,29.0,99.0,57.0,72.0,70.0,84.0,80.0,74.0,85.0,87.0,2013,Sabah


In [ ]:
combined_data.tail()

,Species,January,February,March,April,May,June,July,August,September,October,November,December,Year,location
3409,SotongKurita,0.0,0.0,0.0,0.0,0.0,0.0,6.4741,0.0,0.0,0.0,0.0,0.0,2018,Labuan
3410,SotongMengabang,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2018,Labuan
3411,SotongJarum,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2018,Labuan
3412,SotongTorak,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2018,Labuan
3413,SotongDaun,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2018,Labuan


In [ ]:
combined_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3414 entries, 0 to 3413
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Species    3414 non-null   object
 1   January    3196 non-null   object
 2   February   3414 non-null   object
 3   March      3410 non-null   object
 4   April      3410 non-null   object
 5   May        3410 non-null   object
 6   June       3412 non-null   object
 7   July       3410 non-null   object
 8   August     3412 non-null   object
 9   September  3409 non-null   object
 10  October    3411 non-null   object
 11  November   3413 non-null   object
 12  December   3407 non-null   object
 13  Year       3414 non-null   int64 
 14  location   3414 non-null   object
dtypes: int64(1), object(14)
memory usage: 400.2+ KB


In [ ]:
# Change the datatype of months to int64
month_columns = ['January', 'February', 'March', 'April', 'May', 'June',
                 'July', 'August', 'September', 'October', 'November', 'December']

for col in month_columns:
    combined_data[col] = pd.to_numeric(combined_data[col], errors='coerce').fillna(0).astype('float')

print("Data types after conversion:")
print(combined_data.dtypes)

Data types after conversion:
Species       object
January      float64
February     float64
March        float64
April        float64
May          float64
June         float64
July         float64
August       float64
September    float64
October      float64
November     float64
December     float64
Year           int64
location      object
dtype: object


In [ ]:
# Normalize species names (title case and strip extra whitespace)
combined_data['Species'] = combined_data['Species'].str.strip().str.title()

# Ensure 'Year' is an integer type
combined_data['Year'] = combined_data['Year'].astype(int)

# Sort by Species, location, and Year (ascending)
combined_data = combined_data.sort_values(
    by=['Species', 'location', 'Year'],
    ascending=[True, True, True]
).reset_index(drop=True)

In [ ]:
combined_data

,Species,January,February,March,April,May,June,July,August,September,October,November,December,Year,location
0,Alu-Alu/Kacang-Kacang,66.0000,67.0000,138.0000,185.0000,204.0000,166.0000,122.0000,113.0000,138.0000,121.0000,74.0000,51.0000,2011,East Peninsular Malaysia
1,Alu-Alu/Kacang-Kacang,97.0000,122.0000,156.0000,156.0000,176.0000,150.0000,133.0000,142.0000,153.0000,172.0000,104.0000,76.0000,2012,East Peninsular Malaysia
2,Alu-Alu/Kacang-Kacang,122.0000,147.0000,177.0000,223.0000,168.0000,168.0000,156.0000,124.0000,231.0000,151.0000,113.0000,89.0000,2013,East Peninsular Malaysia
3,Alu-Alu/Kacang-Kacang,77.0000,110.0000,168.0000,189.0000,152.0000,160.0000,145.0000,133.0000,195.0000,158.0000,115.0000,110.0000,2014,East Peninsular Malaysia
4,Alu-Alu/Kacang-Kacang,194.0000,153.0000,175.0000,184.0000,227.0000,216.0000,282.0000,345.0000,222.0000,224.0000,230.0000,146.0000,2015,East Peninsular Malaysia
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3409,Yu,77.5283,85.9187,59.4119,100.8320,89.3182,87.0661,87.5004,99.0643,84.0217,83.7316,69.9604,58.4152,2019,West Peninsular Malaysia
3410,Yu,89.0000,62.0000,66.0000,89.0000,96.0000,63.0000,59.0000,82.0000,82.0000,93.0000,68.0000,85.0000,2020,West Peninsular Malaysia
3411,Yu,102.1480,113.5917,91.7289,95.6381,90.9216,70.9185,83.5217,82.2041,88.6778,102.2421,62.5676,65.3999,2021,West Peninsular Malaysia
3412,Yu,0.0000,85.0000,62.0000,64.0000,80.0000,80.0000,91.0000,113.0000,80.0000,77.0000,74.0000,64.0000,2022,West Peninsular Malaysia


In [ ]:
output_path = "/content/drive/My Drive/FYP_Fish/combined_sorted_data.csv"
combined_data.to_csv(output_path, index=False)

print(f"✅ Sorted data exported to: {output_path}")

✅ Sorted data exported to: /content/drive/My Drive/FYP_Fish/combined_sorted_data.csv


In [ ]:
# Filter the data for the years 2011-2023
filtered_data = combined_data[combined_data['Year'].between(2011, 2023)]

# Step 1: Check species with more than 65 rows
species_row_count = filtered_data.groupby('Species').size()
print(f"Species row count (more than 60 rows):\n{species_row_count}")  # Print the first few rows for inspection
valid_species = species_row_count[species_row_count > 60].index

# Step 2: Filter species with more than 12 rows per location
valid_species_data = filtered_data[filtered_data['Species'].isin(valid_species)]
valid_species_data_location_check = (
    valid_species_data.groupby(['Species', 'location'])
    .size()
    .reset_index(name='location_row_count')
)
# print(f"Valid species with more than 12 rows per location:\n{valid_species_data_location_check.head()}")  # Check if any valid species meet the criteria
valid_species_with_enough_rows = valid_species_data_location_check[
    valid_species_data_location_check['location_row_count'] > 12
]['Species'].unique()

# Step 3: Filter valid species and count zeros across months
final_valid_species_data = valid_species_data[valid_species_data['Species'].isin(valid_species_with_enough_rows)]

# Check the number of valid rows after filtering
# print(f"Filtered data after checking for rows per location:\n{final_valid_species_data.head()}")

# Count zeros for each species across all months
zero_counts = (
    final_valid_species_data
    .groupby('Species')[month_columns]
    .apply(lambda x: (x == 0).sum().sum())  # Count zeros across all months
    .reset_index()  # Reset the index
)

# Check the zero_counts DataFrame to confirm if it's populated
# print(f"Zero counts for species:\n{zero_counts.head()}")  # Check the zero counts data

# Rename the column '0' (after reset_index) to 'zero_count'
zero_counts.rename(columns={0: 'zero_count'}, inplace=True)

# Check again after renaming
# print(f"Zero counts after renaming:\n{zero_counts.head()}")  # Verify renaming has worked

# Step 4: Sort species by least zeros and get top 15 species
top_15_least_zeros = zero_counts.sort_values(by='zero_count').head(15)

# Display the top 15 species with the least number of zeros
print(f"\n\nTop 15 species with least zeros:\n{top_15_least_zeros}")


Species row count (more than 60 rows):
Species
Alu-Alu/Kacang-Kacang     64
Bawalbujang               64
Bawalhitam                64
Bawalputih                64
Bawalselatan              64
                          ..
Udangmerah                64
Udangmerahros             64
Udangputihkecil/Kertas    30
Udangsusu                 64
Yu                        64
Length: 71, dtype: int64


Top 15 species with least zeros:
                  Species  zero_count
12             Kerisibali           0
13              Ketamlaut           0
20                  Selar           0
0   Alu-Alu/Kacang-Kacang           1
6              Bijinangka           1
11                 Kerisi           1
19                Sebelah           1
7       Gelama/Tengkerong           1
8   Kayu/Tongkol/Ayahitam           4
2              Bawalhitam           5
27               Tenggiri           9
3              Bawalputih          11
36                     Yu          11
21            Selarkuning          19
24  

In [ ]:
# Define the folder where the CSV files will be saved
output_folder = "/content/drive/My Drive/FYP_Fish/species_csv_files"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# List of species to extract (same as before)
species_to_extract = [
    'Kerisibali', 'Ketamlaut', 'Selar', 'Alu-Alu/Kacang-Kacang',
    'Bijinangka', 'Kerisi', 'Sebelah', 'Gelama/Tengkerong',
    'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Tenggiri', 'Bawalputih',
    'Yu', 'Selarkuning', 'Siakap'
]

# Loop through each species, filter data, and save to a CSV file
for species in species_to_extract:
    # Filter data for the current species
    species_data = combined_data[combined_data['Species'] == species]

    # Define the filename for the current species
    file_path = os.path.join(output_folder, f"{species.replace('/', '_')}.csv")

    # Save the species data to the CSV file
    species_data.to_csv(file_path, index=False)

    print(f"Saved {species} data to {file_path}")

Saved Kerisibali data to /content/drive/My Drive/FYP_Fish/species_csv_files/Kerisibali.csv
Saved Ketamlaut data to /content/drive/My Drive/FYP_Fish/species_csv_files/Ketamlaut.csv
Saved Selar data to /content/drive/My Drive/FYP_Fish/species_csv_files/Selar.csv
Saved Alu-Alu/Kacang-Kacang data to /content/drive/My Drive/FYP_Fish/species_csv_files/Alu-Alu_Kacang-Kacang.csv
Saved Bijinangka data to /content/drive/My Drive/FYP_Fish/species_csv_files/Bijinangka.csv
Saved Kerisi data to /content/drive/My Drive/FYP_Fish/species_csv_files/Kerisi.csv
Saved Sebelah data to /content/drive/My Drive/FYP_Fish/species_csv_files/Sebelah.csv
Saved Gelama/Tengkerong data to /content/drive/My Drive/FYP_Fish/species_csv_files/Gelama_Tengkerong.csv
Saved Bawalhitam data to /content/drive/My Drive/FYP_Fish/species_csv_files/Bawalhitam.csv
Saved Kayu/Tongkol/Ayahitam data to /content/drive/My Drive/FYP_Fish/species_csv_files/Kayu_Tongkol_Ayahitam.csv
Saved Tenggiri data to /content/drive/My Drive/FYP_Fish/sp

## Check whether there are missing data in extracted species datasets



In [ ]:
extracted_species_folder = "/content/drive/My Drive/FYP_Fish/species_check "

In [ ]:
# Check whether there are missing data in each csv
def check_missing_data(folder_path):
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip', engine='python')
                    missing_count = df.isnull().sum().sum()

                    if missing_count > 0:
                        print(f"Missing values found in {file}: {missing_count} missing cells")
                    else:
                        print(f"No missing values in {file}")

                except Exception as e:
                    print(f"Failed to process {file}: {e}")

In [ ]:
check_missing_data(extracted_species_folder)

## Extract Species that is based on States for Spatial Disaggregation

In [ ]:
input_dir_6 = "/content/drive/My Drive/FYP_Fish/4.4"

Convert 2018, 2019 and 2021 data to csv form

In [ ]:
xlsx_files = {
    "2015": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.xlsx",
    "2018": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.xlsx",
    "2019": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.xlsx",
    "2021": "/content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2021.xlsx"
}

In [ ]:
for year, xlsx_path in xlsx_files.items():
    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(xlsx_path, engine='openpyxl')

    # Create the output CSV file path
    output_file = os.path.join(input_dir_6, f'4.4 - Jadual Pendaratan Ikan Laut {year}.csv')

    # Save the DataFrame to CSV
    df.to_csv(output_file, index=False)

    print(f"🔍 Saved {output_file}")

🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
🔍 Saved /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2021.csv


In [ ]:
clean_csv_files(input_dir_6)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut

In [ ]:
base_dir = "/content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets"

file_paths = {
     "2016": [
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual(Table) 4.4.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_5.csv'),
        os.path.join(base_dir, 'Jadual Pendaratan Ikan Laut 2016', 'Jadual_Page_6.csv')
     ]
}

In [ ]:
def clean_multiple_dirs(file_paths_by_year):
    """Extract unique directories from the file_paths dict and clean CSVs in those directories."""
    visited_dirs = set()
    for year, paths in file_paths_by_year.items():
        for path in paths:
            dir_path = os.path.dirname(path)
            if dir_path not in visited_dirs:
                clean_csv_files(dir_path)
                visited_dirs.add(dir_path)


In [ ]:
clean_multiple_dirs(file_paths)

✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.1.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.6.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.3.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual_Page_1.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.5.csv
✅ Cleaned and saved: /content/drive/My Drive/FYP_Fish/Fish Landings Csv Datasets/Jadual Pendaratan Ikan Laut 2016/Jadual(Table) 4.6.4.csv
✅ Cleaned and saved: /content/drive/My Dri

In [ ]:
# Process each year, merge the files, and save as a CSV
for year, paths in file_paths.items():
    # Read all CSV files for that year
    dfs = [pd.read_csv(path) for path in paths]

    # Count total rows before merging
    column_counts = [len(df.columns) for df in dfs]
    total_columns_before = sum(column_counts)
    row_counts = [len(df) for df in dfs]
    total_rows_before = sum(row_counts)

    # Concatenate all the dataframes for that year
    merged_df = pd.concat(dfs, ignore_index=True)
    total_rows_after = len(merged_df)
    total_columns_after = len(merged_df.columns)

    # Create a filename for the year and save it
    output_file = os.path.join(input_dir_6, f'4.4 - Jadual Pendaratan Ikan Laut {year}.csv')
    merged_df.to_csv(output_file, index=False)

    print(f"\nYear: {year}")
    print(f"Total rows before merging: {total_rows_before}")
    print(f"Total rows after merging: {total_rows_after}")
    print(f"Total columns before merging: {total_columns_before}")
    print(f"Total columns after merging: {total_columns_after}")
    print(f"Saved: {output_file}")


Year: 2016
Total rows before merging: 168
Total rows after merging: 168
Total columns before merging: 63
Total columns after merging: 21
Saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv


In [ ]:
def remove_unnecessary_columns_rows2(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if not file.lower().endswith(".csv"):
                continue

            file_path = os.path.join(root, file)
            match = re.search(r'(20\d{2})', file)
            year = int(match.group(0)) if match else None

            if not year:
                print(f"⚠️ Skipping {file} (no year found)")
                continue

            try:
                df = pd.read_csv(file_path, engine='python', on_bad_lines='skip', header=None)

                # Apply different row slicing based on the year
                if year in list(range(2011, 2015)) + [2016] + [2017] + [2020]:
                    df = df.iloc[3:, 1:]
                elif year in [2015, 2018, 2019]:
                    df = df.iloc[9:, 1:]
                elif year == 2021:
                    df = df.iloc[9:, 1:]
                elif year in [2022, 2023]:
                    df = df.iloc[3:, :]
                else:
                    print(f"ℹ️ No rule for {year} — keeping original")
                    continue

                # Remove rows containing "Jadual 4.4" (case-insensitive)
                df = df[~df.apply(lambda row: row.astype(str).str.contains("jadual 4.4", case=False).any(), axis=1)]

                # Remove empty columns for specific years (2018, 2019, 2021)
                if year in [2015, 2018, 2019, 2021]:
                    df = df.dropna(axis=1, how='all')  # Drop columns with all NaN values

                 # Add an empty first row
                empty_row = pd.DataFrame([[None] * df.shape[1]], columns=df.columns)
                df = pd.concat([empty_row, df], ignore_index=True)

                cleaned_filename = f"{file}"
                save_path = os.path.join(output_dir, cleaned_filename)
                df.to_csv(save_path, index=False, header=False)
                print(f"✅ Saved cleaned file to {save_path}")

            except Exception as e:
                print(f"❌ Error processing {file_path}: {e}")

In [ ]:
# remove header
remove_unnecessary_columns_rows2(input_dir_6, input_dir_6)

✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Saved cleaned file to /content/drive/My Drive/FYP_Fish/4.4/4.4

In [ ]:
def print_column_counts(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')
                    print(f"{file}: {df.shape[1]} columns")
                except Exception as e:
                    print(f"❌ Failed to read {file}: {e}")

In [ ]:
print_column_counts(input_dir_6)

4.4 - Jadual Pendaratan Ikan Laut 2020.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2018.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2012.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2016.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2017.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2013.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2019.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2015.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2011.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2021.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2014.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2010.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2008.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2022.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2006.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2023.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2009.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2007.csv: 21 columns


Calculate the ratio for each species with the total

In [ ]:
def batch_calculate_ratios(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.isfile(file_path) and file.lower().endswith(".csv"):
                calculate_species_ratios(file_path)

def calculate_species_ratios(file_path):
    try:
        # Read the CSV
        df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')

        if df.shape[1] < 3:
            print(f"Skipping: Not enough columns in {file_path}")
            return

        # Identify column indexes
        first_col = df.columns[0]  # Species name
        data_cols = df.columns[1:-1]  # Ratio columns (2nd to second-last)
        total_col = df.columns[-1]  # Total column

        # Convert all data columns to numeric (safely)
        df[data_cols] = df[data_cols].apply(pd.to_numeric, errors='coerce')
        df[total_col] = pd.to_numeric(df[total_col], errors='coerce')

        # Avoid division by zero
        df[data_cols] = df[data_cols].div(df[total_col], axis=0)

        # Save updated CSV
        df.to_csv(file_path, index=False, encoding='utf-8')
        print(f"✅ Ratios calculated and saved: {file_path}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")

In [ ]:
batch_calculate_ratios(input_dir_6)

✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅

Drop subtotal and total columns

In [ ]:
print_column_counts(input_dir_6)

4.4 - Jadual Pendaratan Ikan Laut 2020.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2018.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2012.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2016.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2017.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2013.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2019.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2015.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2011.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2021.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2014.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2010.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2008.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2022.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2006.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2023.csv: 20 columns
4.4 - Jadual Pendaratan Ikan Laut 2009.csv: 21 columns
4.4 - Jadual Pendaratan Ikan Laut 2007.csv: 21 columns


In [ ]:
# Function to drop specific columns
def drop_specific_columns(file_path):
    try:
        df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')

        # Column indexes to drop (0-based): 10, 15, 19, 20
        columns_to_drop = [9, 14, 18, 19]

        # Filter only existing columns to avoid index errors
        existing_cols_to_drop = [df.columns[i] for i in columns_to_drop if i < len(df.columns)]

        # Drop the columns
        df.drop(columns=existing_cols_to_drop, axis=1, inplace=True)

        # Save back
        df.to_csv(file_path, index=False, encoding='utf-8')
        print(f"✅ Dropped columns and saved: {file_path}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")

# Function to process all CSV files in a specified directory
def process_files_drop(input_dir_6):
    # Loop through the directory to find all files
    for root, _, files in os.walk(input_dir_6):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                drop_specific_columns(file_path)

In [ ]:
process_files_drop(input_dir_6)

✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv
✅ Dropped columns and saved: /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv
✅ Dropped columns a

In [ ]:
header = [
    'Species', 'Perlis', 'Kedah', 'Pulau Pinang', 'Perak', 'Selangor', 'Negeri Sembilan',
    'Melaka', 'Johor Barat', 'Kelantan', 'Terengganu', 'Pahang', 'Johor Timur', 'Sarawak',
    'Sabah', 'Labuan'
]

In [ ]:
add_header(input_dir_6, header)

['Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2011.csv',
 'Added header to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2021.csv',


In [ ]:
print_column_counts(input_dir_6)

4.4 - Jadual Pendaratan Ikan Laut 2020.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2018.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2012.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2016.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2017.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2013.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2019.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2015.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2011.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2021.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2014.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2010.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2008.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2022.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2006.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2023.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2009.csv: 16 columns
4.4 - Jadual Pendaratan Ikan Laut 2007.csv: 16 columns


In [ ]:
add_year_column(input_dir_6)

["Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2020.csv with year: 2020",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2018.csv with year: 2018",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2012.csv with year: 2012",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2016.csv with year: 2016",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2017.csv with year: 2017",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2013.csv with year: 2013",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2019.csv with year: 2019",
 "Added 'Year' column to /content/drive/My Drive/FYP_Fish/4.4/4.4 - Jadual Pendaratan Ikan Laut 2015.csv with year: 2015",
 "Added 'Year' c

In [ ]:
combined_data_2 = combine_csv_to_df(input_dir_6)

In [ ]:
combined_data_2

,Species,Perlis,Kedah,Pulau Pinang,Perak,Selangor,Negeri Sembilan,Melaka,Johor Barat,Kelantan,Terengganu,Pahang,Johor Timur,Sarawak,Sabah,Labuan,Year
0,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 15,Unnamed: 16,Unnamed: 17,2020
1,Kebasi,0.0563685636856368,0.566260162601626,0.0493224932249322,0.0016260162601626,NaN,0.0006775067750677,NaN,0.035230352303523,0.0998644986449864,NaN,NaN,NaN,0.0752032520325203,0.1155826558265582,NaN,2020
2,Selangat,NaN,NaN,0.0031616982836495,0.6964769647696477,0.2136404697380307,0.006323396567299,0.0336946702800361,0.0293586269196025,0.0140018066847335,NaN,NaN,0.0031616982836495,0.0004516711833785,NaN,NaN,2020
3,Puput,NaN,0.3325429756965026,0.0002963841138114,0.3549693736415728,0.0921754593953764,0.0014819205690574,NaN,0.0095830863465718,9.879470460383324e-05,NaN,NaN,0.0005927682276229,0.1985773562537048,0.0096818810511756,NaN,2020
4,BeliakMata,NaN,0.0362851475907128,0.0212343950530859,0.3571345233928363,0.3789522809473807,0.0014000700035001,NaN,0.0917045852292614,0.0053669350134173,0.0001166725002916,2.333450005833625e-05,NaN,0.0928713102321782,0.0150507525376268,NaN,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007
2249,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007
2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007
2251,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007


In [ ]:
combined_data_2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2253 entries, 0 to 2252
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Species          2209 non-null   object
 1   Perlis           1223 non-null   object
 2   Kedah            1334 non-null   object
 3   Pulau Pinang     1487 non-null   object
 4   Perak            1929 non-null   object
 5   Selangor         1545 non-null   object
 6   Negeri Sembilan  1051 non-null   object
 7   Melaka           870 non-null    object
 8   Johor Barat      1293 non-null   object
 9   Kelantan         1662 non-null   object
 10  Terengganu       1587 non-null   object
 11  Pahang           1701 non-null   object
 12  Johor Timur      1469 non-null   object
 13  Sarawak          1974 non-null   object
 14  Sabah            1769 non-null   object
 15  Labuan           1369 non-null   object
 16  Year             2253 non-null   int64 
dtypes: int64(1), object(16)
memory us

In [ ]:
# Normalize species names (title case and strip extra whitespace)
combined_data_2['Species'] = combined_data_2['Species'].str.strip().str.title()

# Ensure 'Year' is an integer type
combined_data_2['Year'] = combined_data_2['Year'].astype(int)

# Sort by Species, location, and Year (ascending)
combined_data_2 = combined_data_2.sort_values(
    by=['Species', 'Year'],
    ascending=[True, True]
).reset_index(drop=True)

In [ ]:
# Filter the data for the years 2011-2023
filtered_data = combined_data_2[combined_data['Year'].between(2011, 2023)]

# Step 1: Check species with more than 65 rows
species_row_count = filtered_data.groupby('Species').size()
print(f"Species row count (more than 12 rows):\n{species_row_count}")  # Print the first few rows for inspection
valid_species = species_row_count[species_row_count > 12].index

# Step 2: Filter species with more than 12 rows per location
valid_species_data = filtered_data[filtered_data['Species'].isin(valid_species)]
valid_species_data_location_check = (
    valid_species_data.groupby(['Species'])
    .size()
    .reset_index(name='location_row_count')
)
# print(f"Valid species with more than 12 rows per location:\n{valid_species_data_location_check.head()}")  # Check if any valid species meet the criteria
valid_species_with_enough_rows = valid_species_data_location_check[
    valid_species_data_location_check['location_row_count'] > 12
]['Species'].unique()

# Step 3: Filter valid species and count zeros across months
final_valid_species_data = valid_species_data[valid_species_data['Species'].isin(valid_species_with_enough_rows)]

# Check the number of valid rows after filtering
# print(f"Filtered data after checking for rows per location:\n{final_valid_species_data.head()}")

# Count zeros for each species across all months
zero_counts = (
    final_valid_species_data
    .groupby('Species')
    .apply(lambda x: (x == 0).sum().sum())  # Count zeros across all months
    .reset_index()  # Reset the index
)

# Check the zero_counts DataFrame to confirm if it's populated
# print(f"Zero counts for species:\n{zero_counts.head()}")  # Check the zero counts data

# Rename the column '0' (after reset_index) to 'zero_count'
zero_counts.rename(columns={0: 'zero_count'}, inplace=True)

# Check again after renaming
# print(f"Zero counts after renaming:\n{zero_counts.head()}")  # Verify renaming has worked

# Step 4: Sort species by least zeros and get top 15 species
top_15_least_zeros = zero_counts.sort_values(by='zero_count').head(15)

# Display the top 15 species with the least number of zeros
print(f"\n\nTop 15 species with least zeros:\n{top_15_least_zeros}")


Species row count (more than 12 rows):
Species
Aji-Aji                   13
Alu-Alu/Kacang-Kacang     13
Aruantasek                 2
Aruantasik                11
Baji-Baji                 13
                          ..
Udangputihkecil/Kertas     7
Udangputihsedang          11
Udangsusu                 13
Unnamed: 0                13
Yu                        13
Length: 234, dtype: int64


Top 15 species with least zeros:
                  Species  zero_count
0                 Aji-Aji           0
1   Alu-Alu/Kacang-Kacang           0
2               Baji-Baji           0
3             Bawalbujang           0
4              Bawalhitam           0
5              Bawalputih           0
6            Bawalselatan           0
7             Bawaltambak           0
8                   Bayan           0
9              Bijinangka           0
10         Bilis/Bungaair           0
11            Bulan-Bulan           0
12                   Bulu           0
13               Buluayam           0
14 

<ipython-input-886-3692306747>:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filtered_data = combined_data_2[combined_data['Year'].between(2011, 2023)]
<ipython-input-886-3692306747>:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x == 0).sum().sum())  # Count zeros across all months


In [ ]:
# Define the folder where the CSV files will be saved
output_folder = "/content/drive/My Drive/FYP_Fish/location_csv_files"

# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# List of species to extract (same as before)
species_to_extract = [
    'Kerisibali', 'Ketamlaut', 'Selar', 'Alu-Alu/Kacang-Kacang',
    'Bijinangka', 'Kerisi', 'Sebelah', 'Gelama/Tengkerong',
    'Bawalhitam', 'Kayu/Tongkol/Ayahitam', 'Tenggiri', 'Bawalputih',
    'Yu', 'Selarkuning', 'Siakap'
]

# Loop through each species, filter data, and save to a CSV file
for species in species_to_extract:
    # Filter data for the current species
    species_data = combined_data_2[combined_data_2['Species'] == species]

    # Define the filename for the current species
    file_path = os.path.join(output_folder, f"{species.replace('/', '_')}.csv")

    # Save the species data to the CSV file
    species_data.to_csv(file_path, index=False)

    print(f"Saved {species} data to {file_path}")

Saved Kerisibali data to /content/drive/My Drive/FYP_Fish/location_csv_files/Kerisibali.csv
Saved Ketamlaut data to /content/drive/My Drive/FYP_Fish/location_csv_files/Ketamlaut.csv
Saved Selar data to /content/drive/My Drive/FYP_Fish/location_csv_files/Selar.csv
Saved Alu-Alu/Kacang-Kacang data to /content/drive/My Drive/FYP_Fish/location_csv_files/Alu-Alu_Kacang-Kacang.csv
Saved Bijinangka data to /content/drive/My Drive/FYP_Fish/location_csv_files/Bijinangka.csv
Saved Kerisi data to /content/drive/My Drive/FYP_Fish/location_csv_files/Kerisi.csv
Saved Sebelah data to /content/drive/My Drive/FYP_Fish/location_csv_files/Sebelah.csv
Saved Gelama/Tengkerong data to /content/drive/My Drive/FYP_Fish/location_csv_files/Gelama_Tengkerong.csv
Saved Bawalhitam data to /content/drive/My Drive/FYP_Fish/location_csv_files/Bawalhitam.csv
Saved Kayu/Tongkol/Ayahitam data to /content/drive/My Drive/FYP_Fish/location_csv_files/Kayu_Tongkol_Ayahitam.csv
Saved Tenggiri data to /content/drive/My Drive/F

Replace missing values in datasets with zeros

In [ ]:
extracted_location_folder = "/content/drive/My Drive/FYP_Fish/location_csv_files"

In [ ]:
check_missing_data(extracted_location_folder)

Missing values found in Bawalputih.csv: 1 missing cells
Missing values found in Bawalhitam.csv: 11 missing cells
Missing values found in Selarkuning.csv: 33 missing cells
Missing values found in Sebelah.csv: 10 missing cells
Missing values found in Yu.csv: 7 missing cells
Missing values found in Kayu_Tongkol_Ayahitam.csv: 34 missing cells
No missing values in Gelama_Tengkerong.csv
Missing values found in Tenggiri.csv: 2 missing cells
Missing values found in Selar.csv: 16 missing cells
Missing values found in Siakap.csv: 25 missing cells
Missing values found in Ketamlaut.csv: 18 missing cells
Missing values found in Alu-Alu_Kacang-Kacang.csv: 5 missing cells
Missing values found in Kerisi.csv: 20 missing cells
Missing values found in Kerisibali.csv: 60 missing cells
Missing values found in Bijinangka.csv: 18 missing cells


In [ ]:
for filename in os.listdir(extracted_location_folder):
    if filename.endswith(".csv"):
        filepath = os.path.join(extracted_location_folder, filename)

        # Read CSV
        df = pd.read_csv(filepath)

        # Replace empty strings and NaNs through
        df.replace('', 0, inplace=True)
        df.fillna(0, inplace=True)

        # Save the modified file back
        df.to_csv(filepath, index=False)

print("Empty values replaced with 0 in all CSV files.")

Empty values replaced with 0 in all CSV files.


In [ ]:
check_missing_data(extracted_location_folder)

No missing values in Bawalputih.csv
No missing values in Bawalhitam.csv
No missing values in Selarkuning.csv
No missing values in Sebelah.csv
No missing values in Yu.csv
No missing values in Kayu_Tongkol_Ayahitam.csv
No missing values in Gelama_Tengkerong.csv
No missing values in Tenggiri.csv
No missing values in Selar.csv
No missing values in Siakap.csv
No missing values in Ketamlaut.csv
No missing values in Alu-Alu_Kacang-Kacang.csv
No missing values in Kerisi.csv
No missing values in Kerisibali.csv
No missing values in Bijinangka.csv


# Spatial Disaggregation using proxy-based method based on month, state and year

In [ ]:
monthly_data_folder = "/content/drive/My Drive/FYP_Fish/species_files"
state_ratio_folder = "/content/drive/My Drive/FYP_Fish/location_csv_files"
output_folder = "/content/drive/My Drive/FYP_Fish/output_disaggregated"

os.makedirs(output_folder, exist_ok=True)

In [ ]:
# Define a mapping from states to regions
state_to_region = {
    'Perlis': 'West Peninsular Malaysia',
    'Kedah': 'West Peninsular Malaysia',
    'Pulau Pinang': 'West Peninsular Malaysia',
    'Perak': 'West Peninsular Malaysia',
    'Selangor': 'West Peninsular Malaysia',
    'Negeri Sembilan': 'West Peninsular Malaysia',
    'Melaka': 'West Peninsular Malaysia',
    'Johor Barat': 'West Peninsular Malaysia',
    'Kelantan': 'East Peninsular Malaysia',
    'Terengganu': 'East Peninsular Malaysia',
    'Pahang': 'East Peninsular Malaysia',
    'Johor Timur': 'East Peninsular Malaysia',
    'Sarawak': 'Sarawak', # No need to iterate
    'Sabah': 'Sabah', # No need to iterate
    'Labuan': 'Labuan' # No need to iterate
}


In [ ]:
# Group states by region (excluding Sarawak, Sabah, Labuan)
states_by_region = {}
for state, region in state_to_region.items():
    if region in ['Sarawak', 'Sabah', 'Labuan']:
        continue
    if region not in states_by_region:
        states_by_region[region] = []
    states_by_region[region].append(state)

In [ ]:
check_missing_data(state_ratio_folder)

No missing values in Bawalputih.csv
No missing values in Bawalhitam.csv
No missing values in Selarkuning.csv
No missing values in Sebelah.csv
No missing values in Yu.csv
No missing values in Kayu_Tongkol_Ayahitam.csv
No missing values in Gelama_Tengkerong.csv
No missing values in Tenggiri.csv
No missing values in Selar.csv
No missing values in Siakap.csv
No missing values in Ketamlaut.csv
No missing values in Alu-Alu_Kacang-Kacang.csv
No missing values in Kerisi.csv
No missing values in Kerisibali.csv
No missing values in Bijinangka.csv


In [ ]:
# Group states by region
states_by_region = {}
for state, region in state_to_region.items():
    states_by_region.setdefault(region, []).append(state)

# Get all species filenames
species_files = [f for f in os.listdir(monthly_data_folder) if f.endswith('.csv')]

# Process each file
for filename in species_files:
    monthly_path = os.path.join(monthly_data_folder, filename)
    ratio_path = os.path.join(state_ratio_folder, filename)

    if not os.path.exists(ratio_path):
        print(f"Missing ratio file for {filename}. Skipping.")
        continue

    # Load data
    monthly_df = pd.read_csv(monthly_path)
    ratio_df = pd.read_csv(ratio_path)

    # Clean and preprocess
    monthly_df.replace('', 0, inplace=True)
    monthly_df.fillna(0, inplace=True)
    ratio_df.fillna(0, inplace=True)

    # Ensure numeric monthly values
    monthly_df[month_columns] = monthly_df[month_columns].apply(pd.to_numeric, errors='coerce').fillna(0)

    # Remove rows with no data
    monthly_df['Year'] = pd.to_numeric(monthly_df['Year'], errors='coerce')
    monthly_df.dropna(subset=['Year'], inplace=True)
    monthly_df['Year'] = monthly_df['Year'].astype(int)
    monthly_df = monthly_df[monthly_df[month_columns].sum(axis=1) > 0]

    # Extract species name
    species_name = monthly_df['species'].iloc[0] if 'species' in monthly_df.columns else filename.replace('.csv', '')

    all_state_monthly = []

    for year in range(2011, 2024):
        if year not in ratio_df['Year'].values:
            continue

        state_ratios = ratio_df[ratio_df['Year'] == year].iloc[0]
        year_data = monthly_df[monthly_df['Year'] == year]

        if year_data.empty:
            continue

        state_monthly_catches = pd.DataFrame({'Year': [year] * 12, 'Month': list(range(1, 13))})

        for state in state_to_region:
            region = state_to_region[state]
            region_data = year_data[year_data['location'].str.lower() == region.lower()]

            if region in ['Sabah', 'Sarawak', 'Labuan']:
                if region_data.empty:
                    state_monthly_catches[state] = [0] * 12
                else:
                    state_monthly_catches[state] = [
                        region_data[month].values[0] for month in month_columns
                    ]
                continue

            region_states = states_by_region[region]
            region_total_ratio = sum(state_ratios[s] for s in region_states if s in state_ratios)
            ratio = state_ratios[state] / region_total_ratio if region_total_ratio > 0 else 0

            if region_data.empty:
                state_monthly_catches[state] = [0] * 12
                continue

            state_monthly_catches[state] = [
                region_data[month].values[0] * ratio for month in month_columns
            ]

        # Add species column
        state_monthly_catches.insert(0, 'Species', species_name)
        all_state_monthly.append(state_monthly_catches)

    # Save output
    if all_state_monthly:
        result_df = pd.concat(all_state_monthly, ignore_index=True)
        output_path = os.path.join(output_folder, filename.replace('.csv', '_disaggregated.csv'))
        result_df.to_csv(output_path, index=False)
        print(f"✅ Saved disaggregated file: {output_path}")
    else:
        print(f"⚠️ No valid data for {filename}. Skipped.")

✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Alu-Alu_Kacang-Kacang_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Bawalhitam_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Bawalputih_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Bijinangka_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Gelama_Tengkerong_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Kayu_Tongkol_Ayahitam_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Kerisi_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Kerisibali_disaggregated.csv
✅ Saved disaggregated file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Ketam

# Handle Extracted Fishing Gear Dataset

In [ ]:
fishing_gear_folder = "/content/drive/My Drive/FYP_Fish/fishing_gear_files"

In [ ]:
add_location_column(fishing_gear_folder)

Processing file: cleaned_species_sarawak.csv
cleaned_species_sarawak.csv
Updated with location: Sarawak

Processing file: cleaned_species_east.csv
cleaned_species_east.csv
Updated with location: East Peninsular Malaysia

Processing file: cleaned_species_labuan.csv
cleaned_species_labuan.csv
Updated with location: Labuan

Processing file: cleaned_species_sabah - cleaned_species_sabah2.csv
cleaned_species_sabah - cleaned_species_sabah2.csv
Updated with location: Sabah

Processing file: cleaned_species_west.csv
cleaned_species_west.csv
Updated with location: West Peninsular Malaysia

Processing file: cleaned_species_sabah.csv
cleaned_species_sabah.csv
Updated with location: Sabah



In [ ]:
cleaned_fishing_gear_folder = "/content/drive/My Drive/FYP_Fish/final_gear"

In [ ]:
# Use glob to find all .xlsx files
xlsx_files = glob.glob(f'{cleaned_fishing_gear_folder}')

print("Found files:")
print(xlsx_files)  # Debug: see what it found

Found files:
['/content/drive/My Drive/FYP_Fish/final_gear']


Sort gear based on species and year

In [ ]:
def sort_gear_species_year(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)

                try:
                    df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')

                    if df.shape[1] < 5:
                        print(f"⚠️ Skipping (not enough columns): {file_path}")
                        continue

                    first_col = df.columns[0]          # Assume species name
                    data_cols = df.columns[1:-3]      # Data columns to keep
                    total_col = df.columns[-3]        # Total column

                    # Convert ratio and total columns to numeric
                    df[data_cols] = df[data_cols].apply(pd.to_numeric, errors='coerce')
                    df[total_col] = pd.to_numeric(df[total_col], errors='coerce')

                    # Try to find the 'Year' column (case-insensitive)
                    year_col = next((col for col in df.columns if col.lower() == 'year'), None)

                    # Sort the DataFrame
                    if year_col:
                        df.sort_values(by=[first_col, year_col], inplace=True)
                    else:
                        df.sort_values(by=[first_col], inplace=True)

                    # Save the sorted DataFrame
                    df.to_csv(file_path, index=False, encoding='utf-8')
                    print(f"✅ Sorted: {file_path}")

                except Exception as e:
                    print(f"❌ Error processing {file_path}: {e}")


In [ ]:
sort_gear_species_year(cleaned_fishing_gear_folder)

✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/labuan.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/sarawak.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/sabah.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/east.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/west.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/combined_gear_data.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Alu-Alu_Kacang-Kacang.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalhitam.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalputih.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bijinangka.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Gelama_Tengkerong.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Kayu_Tongkol_Ayahitam.csv
✅ Sorted: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Kerisi.csv
✅ Sorted: /conten

## Handle missing data for labuan

In [ ]:
fishing_gear_2018 = pd.read_csv("/content/drive/My Drive/FYP_Fish/cleaned_species_labuan - cleaned_species_labuan.csv")
fishing_gear_2018.head()

,Species,Trawl Nets,Fish Purse Seines,Anchovy Purse Seines,Other Seines,Drift/Gill Nets,Lift Nets,Stationary Traps,Portable Traps,Hooks & Lines,Bag Nets,Barrier Nets,Push/Scoop Nets,ShellfishCollection,Miscellaneous,Total,Year,location
0,Kerisibali,168.0,0.0,0,0,85.0,5,0,21.0,492.0,0,0,0,0,0,771.0,2011,Labuan
1,Ketamlaut,231.0,0.0,0,0,0.0,0,0,0.0,0.0,0,0,0,0,0,231.0,2011,Labuan
2,Bijinangka,212.0,0.0,0,0,0.0,4,0,0.0,21.0,0,0,0,0,0,237.0,2011,Labuan
3,Kerisi,817.0,0.0,0,0,0.0,0,0,0.0,0.0,0,0,0,0,0,817.0,2011,Labuan
4,Sebelah,185.0,0.0,0,0,0.0,1,0,1.0,14.0,0,0,0,0,0,202.0,2011,Labuan


In [ ]:
# Ensure proper types
fishing_gear_2018['Year'] = fishing_gear_2018['Year'].astype(int)

# List of gear type columns to use for melting
gear_columns = [
    'Trawl Nets', 'Fish Purse Seines', 'Anchovy Purse Seines', 'Other Seines',
    'Drift/Gill Nets', 'Lift Nets', 'Stationary Traps', 'Portable Traps',
    'Hooks & Lines', 'Bag Nets', 'Barrier Nets', 'Push/Scoop Nets',
    'ShellfishCollection', 'Miscellaneous'
]

# Create a full range of years for each (Species, Location) pair
full_years = list(range(2011, 2024))  # 2011 to 2023

# Expand data to ensure each Species-Location has all years
df_full = (
    fishing_gear_2018.set_index(['Species', 'location', 'Year'])
      .reindex(pd.MultiIndex.from_product(
          [fishing_gear_2018['Species'].unique(), fishing_gear_2018['location'].unique(), full_years],
          names=['Species', 'location', 'Year']
      ))
      .reset_index()
)

# Interpolate the gear columns only
df_full[gear_columns] = df_full.groupby(['Species', 'location'])[gear_columns].transform(lambda group: group.interpolate(method='linear'))

# Recalculate the Total
df_full['Total'] = df_full[gear_columns].sum(axis=1)

df_full.tail()

,Species,location,Year,Trawl Nets,Fish Purse Seines,Anchovy Purse Seines,Other Seines,Drift/Gill Nets,Lift Nets,Stationary Traps,Portable Traps,Hooks & Lines,Bag Nets,Barrier Nets,Push/Scoop Nets,ShellfishCollection,Miscellaneous,Total
190,Selarkuning,Labuan,2019,283.5716,83.7811,0.0,0.0,145.2053,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,512.5580
191,Selarkuning,Labuan,2020,245.0000,87.0000,0.0,0.0,181.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,513.0000
192,Selarkuning,Labuan,2021,264.3001,97.5082,0.0,0.0,126.9053,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,488.7136
193,Selarkuning,Labuan,2022,0.0000,291.0000,182.0,0.0,0.0000,159.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,632.0000
194,Selarkuning,Labuan,2023,0.0000,291.0000,134.0,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,425.0000


In [ ]:
# List of columns
cols = df_full.columns.tolist()

# Remove 'Year' and 'location' from the list first
cols.remove('Year')
cols.remove('location')

# Append 'Total' if it exists (just in case)
if 'Total' in cols:
    cols.remove('Total')

# Now construct the new order:
# Species + gear_columns + Total + Year + location
new_order = ['Species'] + cols[1:] + ['Total', 'Year', 'location']

# Reorder DataFrame
df_full = df_full[new_order]

df_full.head()

,Species,Trawl Nets,Fish Purse Seines,Anchovy Purse Seines,Other Seines,Drift/Gill Nets,Lift Nets,Stationary Traps,Portable Traps,Hooks & Lines,Bag Nets,Barrier Nets,Push/Scoop Nets,ShellfishCollection,Miscellaneous,Total,Year,location
0,Kerisibali,168.0,0.0,0.0,0.0,85.0,5.0,0.0,21.0,492.0,0.0,0.0,0.0,0.0,0.0,771.0,2011,Labuan
1,Kerisibali,154.0,0.0,0.0,0.0,108.0,6.0,0.0,20.0,828.0,0.0,0.0,0.0,0.0,0.0,1116.0,2012,Labuan
2,Kerisibali,134.0,0.0,0.0,0.0,88.0,6.0,0.0,16.0,1166.0,0.0,0.0,0.0,0.0,0.0,1410.0,2013,Labuan
3,Kerisibali,83.0,0.0,0.0,0.0,56.0,9.0,0.0,23.0,534.0,0.0,0.0,0.0,0.0,0.0,705.0,2014,Labuan
4,Kerisibali,198.0,0.0,0.0,0.0,45.0,7.0,0.0,3.0,371.0,0.0,0.0,0.0,0.0,0.0,624.0,2015,Labuan


In [ ]:
# Save or display
df_full.to_csv("/content/drive/My Drive/FYP_Fish/final_gear/labuan.csv", index=False)

## Calculate the ratio for each fishing gear

In [ ]:
def batch_gear_calculate_ratios(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.isfile(file_path) and file.lower().endswith(".csv"):
                calculate_gear_ratios(file_path)

def calculate_gear_ratios(file_path):
    try:
        # Read the CSV
        df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')

        if df.shape[1] < 3:
            print(f"Skipping: Not enough columns in {file_path}")
            return

        # Identify column indexes
        first_col = df.columns[0]  # Species name
        data_cols = df.columns[1:-3]  # Ratio columns (2nd to second-last)
        total_col = df.columns[-3]  # Total column

        # Convert all data columns to numeric (safely)
        df[data_cols] = df[data_cols].apply(pd.to_numeric, errors='coerce')
        df[total_col] = pd.to_numeric(df[total_col], errors='coerce')

        # Fix the total column by summing up data_cols into total_cols
        df[total_col] = df[data_cols].sum(axis=1)

        # Avoid division by zero
        df[data_cols] = df[data_cols].div(df[total_col], axis=0)

        # Save updated CSV
        df.to_csv(file_path, index=False, encoding='utf-8')
        print(f"✅ Ratios calculated and saved: {file_path}")

    except Exception as e:
        print(f"❌ Error processing {file_path}: {e}")

In [ ]:
batch_gear_calculate_ratios(cleaned_fishing_gear_folder)

✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/labuan.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/sarawak.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/sabah.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/east.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/west.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/combined_gear_data.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Alu-Alu_Kacang-Kacang.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalhitam.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalputih.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bijinangka.csv
✅ Ratios calculated and saved: /content/drive/My Drive/FYP_Fi

Remove total columns

In [ ]:
def remove_total_column(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            file_path = os.path.join(root, file)
            if os.path.isfile(file_path) and file.lower().endswith(".csv"):
                try:
                    df = pd.read_csv(file_path, encoding='utf-8', engine='python', on_bad_lines='skip')

                    if df.shape[1] < 3:
                        print(f"⚠️ Skipping (not enough columns): {file_path}")
                        continue

                    # Identify total column (3rd from the end)
                    total_col = df.columns[-3]
                    df.drop(columns=total_col, inplace=True)

                    # Save updated CSV
                    df.to_csv(file_path, index=False, encoding='utf-8')
                    print(f"✅ Removed total column: {file_path}")

                except Exception as e:
                    print(f"❌ Error processing {file_path}: {e}")


In [ ]:
remove_total_column(cleaned_fishing_gear_folder)

✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/labuan.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/sarawak.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/sabah.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/east.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/west.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/combined_gear_data.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Alu-Alu_Kacang-Kacang.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalhitam.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalputih.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bijinangka.csv
✅ Removed total column: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Gelama_Tengkerong.csv
✅ Removed total column: /conte

Combine all the files into a file

In [ ]:
combined_gear_data = combine_csv_to_df(cleaned_fishing_gear_folder)

In [ ]:
combined_gear_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1950 entries, 0 to 1949
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Species               1950 non-null   object 
 1   Trawl Nets            1906 non-null   float64
 2   Fish Purse Seines     1926 non-null   float64
 3   Anchovy Purse Seines  1934 non-null   float64
 4   Other Seines          1932 non-null   float64
 5   Drift/Gill Nets       1914 non-null   float64
 6   Lift Nets             1932 non-null   float64
 7   Stationary Traps      1934 non-null   float64
 8   Portable Traps        1934 non-null   float64
 9   Hooks & Lines         1934 non-null   float64
 10  Bag Nets              1934 non-null   float64
 11  Barrier Nets          1934 non-null   float64
 12  Push/Scoop Nets       187 non-null    float64
 13  ShellfishCollection   187 non-null    float64
 14  Miscellaneous         187 non-null    float64
 15  Year                 

In [ ]:
combined_gear_data.head()

,Species,Trawl Nets,Fish Purse Seines,Anchovy Purse Seines,Other Seines,Drift/Gill Nets,Lift Nets,Stationary Traps,Portable Traps,Hooks & Lines,Bag Nets,Barrier Nets,Push/Scoop Nets,ShellfishCollection,Miscellaneous,Year,location,Location
0,Kerisibali,0.217899,0.0,0.0,0.0,0.110246,0.006485,0.0,0.027237,0.638132,0.0,0.0,0.0,0.0,0.0,2011,Labuan,NaN
1,Kerisibali,0.137993,0.0,0.0,0.0,0.096774,0.005376,0.0,0.017921,0.741935,0.0,0.0,0.0,0.0,0.0,2012,Labuan,NaN
2,Kerisibali,0.095035,0.0,0.0,0.0,0.062411,0.004255,0.0,0.011348,0.826950,0.0,0.0,0.0,0.0,0.0,2013,Labuan,NaN
3,Kerisibali,0.117730,0.0,0.0,0.0,0.079433,0.012766,0.0,0.032624,0.757447,0.0,0.0,0.0,0.0,0.0,2014,Labuan,NaN
4,Kerisibali,0.317308,0.0,0.0,0.0,0.072115,0.011218,0.0,0.004808,0.594551,0.0,0.0,0.0,0.0,0.0,2015,Labuan,NaN


In [ ]:
# Step 1: Remove duplicate columns
combined_gear_data = combined_gear_data.loc[:, ~combined_gear_data.columns.duplicated()]

# Step 2: Standardize column names (optional but helpful)
combined_gear_data.columns = [col.strip().title() for col in combined_gear_data.columns]

# Step 3: Verify available columns
print("✅ Cleaned columns:", combined_gear_data.columns.tolist())

# Step 4: Define and apply sorting
sort_cols = []
if 'Species' in combined_gear_data.columns:
    sort_cols.append('Species')
if 'Location' in combined_gear_data.columns:
    sort_cols.append('Location')
if 'Year' in combined_gear_data.columns:
    sort_cols.append('Year')

if sort_cols:
    combined_gear_data.sort_values(by=sort_cols, inplace=True)
    print("✅ Data sorted by:", sort_cols)
else:
    print("⚠️ Required sorting columns not found.")


✅ Cleaned columns: ['Species', 'Trawl Nets', 'Fish Purse Seines', 'Anchovy Purse Seines', 'Other Seines', 'Drift/Gill Nets', 'Lift Nets', 'Stationary Traps', 'Portable Traps', 'Hooks & Lines', 'Bag Nets', 'Barrier Nets', 'Push/Scoop Nets', 'Shellfishcollection', 'Miscellaneous', 'Year', 'Location']
✅ Data sorted by: ['Species', 'Location', 'Year']


<ipython-input-915-4256353164>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_gear_data.sort_values(by=sort_cols, inplace=True)


In [ ]:
output_path = "/content/drive/My Drive/FYP_Fish/final_gear/combined_gear_data.csv"
combined_gear_data.to_csv(output_path, index=False)

print(f"✅ Sorted data exported to: {output_path}")

✅ Sorted data exported to: /content/drive/My Drive/FYP_Fish/final_gear/combined_gear_data.csv


## Seperate into the combined_gear_data into different csv based on species and saved it in another folder

In [ ]:
def split_and_save_by_species(combined_df, output_folder):
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)

    # Ensure 'Species' column exists (case-insensitive)
    species_col = next((col for col in combined_df.columns if col.lower() == 'species'), None)
    if not species_col:
        print("❌ 'Species' column not found in DataFrame.")
        return

    # Group by species and save each to a separate CSV
    for species, group_df in combined_df.groupby(species_col):
        # Sanitize species name for file naming
        safe_species = "".join(c if c.isalnum() or c in (' ', '_', '-') else "_" for c in str(species)).strip().replace(" ", "_")
        output_path = os.path.join(output_folder, f"{safe_species}.csv")

        group_df.to_csv(output_path, index=False)
        print(f"✅ Saved: {output_path}")

In [ ]:
output_species_folder = "/content/drive/My Drive/FYP_Fish/final_gear/by_species"
split_and_save_by_species(combined_gear_data, output_species_folder)

✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Alu-Alu_Kacang-Kacang.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalhitam.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bawalputih.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Bijinangka.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Gelama_Tengkerong.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Kayu_Tongkol_Ayahitam.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Kerisi.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Kerisibali.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Ketamlaut.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Sebelah.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Selar.csv
✅ Saved: /content/drive/My Drive/FYP_Fish/final_gear/by_species/Selarkuning.csv
✅ Saved: /content/drive/My Drive/FY

# Merge both Datasets

In [ ]:
# Define state-to-region mapping
state_to_region = {
    'Perlis': 'West Peninsular Malaysia',
    'Kedah': 'West Peninsular Malaysia',
    'Pulau Pinang': 'West Peninsular Malaysia',
    'Perak': 'West Peninsular Malaysia',
    'Selangor': 'West Peninsular Malaysia',
    'Negeri Sembilan': 'West Peninsular Malaysia',
    'Melaka': 'West Peninsular Malaysia',
    'Johor Barat': 'West Peninsular Malaysia',
    'Kelantan': 'East Peninsular Malaysia',
    'Terengganu': 'East Peninsular Malaysia',
    'Pahang': 'East Peninsular Malaysia',
    'Johor Timur': 'East Peninsular Malaysia',
    'Sarawak': 'Sarawak',
    'Sabah': 'Sabah',
    'Labuan': 'Labuan'
}

In [ ]:
# Set folder paths
folder_landings = "/content/drive/My Drive/FYP_Fish/copy_output_disaggregated"
folder_gear = "/content/drive/My Drive/FYP_Fish/gear_by_species"
folder_output = "/content/drive/My Drive/FYP_Fish/monthly_landings_state_gear"

# Create output folder
os.makedirs(folder_output, exist_ok=True)

# Get common file names
common_files = set(os.listdir(folder_landings)).intersection(os.listdir(folder_gear))

## Data imputation for monthly_landings_by_state dataset

Data Imputation through Kalman Filter Algorithm

In [ ]:
def kalman_impute(series):
    # Convert to numeric and handle empty strings
    series = pd.to_numeric(series, errors='coerce')

    if series.dropna().empty:
        return series.fillna(0)  # fallback for completely empty column

    # Fill forward/backward temporarily so Kalman can start
    filled_series = series.fillna(method='pad').fillna(method='bfill')

    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        initial_state_mean=filled_series.iloc[0],
        initial_state_covariance=1,
        observation_covariance=1,
        transition_covariance=0.01
    )

    state_means, _ = kf.smooth(filled_series.values)
    return pd.Series(state_means.flatten(), index=series.index)

In [ ]:
def data_imputation_monthly_landings(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                print(f"Processing file: {file_path}")

                # Read CSV to DataFrame
                df = pd.read_csv(file_path)

                # Define metadata columns which should NOT be imputed
                metadata_cols = ['Species', 'Year', 'Month']  # adjust as needed

                # Columns to impute (all except metadata)
                data_cols = [col for col in df.columns if col not in metadata_cols]

                # Apply Kalman imputation on each data column
                for col in data_cols:
                    df[col] = kalman_impute(df[col])

                # Save imputed data with species-based filename
                species_name = df['Species'].iloc[0]
                new_filename = f"{species_name}.csv"
                out_file_path = os.path.join(output_dir, new_filename)
                df.to_csv(out_file_path, index=False)
                print(f"✅ Saved imputed data to: {out_file_path}")

In [ ]:
data_to_impute = "/content/drive/My Drive/FYP_Fish/output_disaggregated"
data_imputation_monthly_landings(data_to_impute, folder_landings)

Processing file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Alu-Alu_Kacang-Kacang_disaggregated.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved imputed data to: /content/drive/My Drive/FYP_Fish/copy_output_disaggregated/Alu-Alu_Kacang-Kacang.csv
Processing file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Bawalhitam_disaggregated.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved imputed data to: /content/drive/My Drive/FYP_Fish/copy_output_disaggregated/Bawalhitam.csv
Processing file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Bawalputih_disaggregated.csv


...

✅ Saved imputed data to: /content/drive/My Drive/FYP_Fish/copy_output_disaggregated/Siakap.csv
Processing file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Tenggiri_disaggregated.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved imputed data to: /content/drive/My Drive/FYP_Fish/copy_output_disaggregated/Tenggiri.csv
Processing file: /content/drive/My Drive/FYP_Fish/output_disaggregated/Yu_disaggregated.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved imputed data to: /content/drive/My Drive/FYP_Fish/copy_output_disaggregated/Yu.csv


In [ ]:
def data_imputation_monthly_landings(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                print(f"📂 Processing: {file_path}")

                try:
                    df = pd.read_csv(file_path)

                    metadata_cols = ['Species', 'Year', 'Location']
                    data_cols = [col for col in df.columns if col not in metadata_cols]

                    for col in data_cols:
                        df[col] = kalman_impute(df[col])

                    out_file_path = os.path.join(output_dir, file)
                    df.to_csv(out_file_path, index=False)

                    print(f"✅ Saved: {out_file_path}")

                except Exception as e:
                    print(f"❌ Error processing {file_path}: {e}")


In [ ]:
folder_gear_imputed = "/content/drive/My Drive/FYP_Fish/gear_by_species/imputed"
data_imputation_monthly_landings(folder_gear, folder_gear_imputed)

📂 Processing: /content/drive/My Drive/FYP_Fish/gear_by_species/Yu.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Yu.csv
📂 Processing: /content/drive/My Drive/FYP_Fish/gear_by_species/Tenggiri.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Tenggiri.csv
📂 Processing: /content/drive/My Drive/FYP_Fish/gear_by_species/Siakap.csv


...

✅ Saved: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Bawalputih.csv
📂 Processing: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Bawalhitam.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Bawalhitam.csv
📂 Processing: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Alu-Alu_Kacang-Kacang.csv


<ipython-input-921-1948708361>:9: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  filled_series = series.fillna(method='pad').fillna(method='bfill')


✅ Saved: /content/drive/My Drive/FYP_Fish/gear_by_species/imputed/Alu-Alu_Kacang-Kacang.csv


In [ ]:
for filename in common_files:
    print(f"Processing: {filename}")

    path_landings = os.path.join(folder_landings, filename)
    path_gear = os.path.join(folder_gear_imputed, filename)

    ext = os.path.splitext(filename)[1].lower()

    try:
        if ext == ".csv":
            monthly_df = pd.read_csv(path_landings)
            gear_df = pd.read_csv(path_gear)
        elif ext in [".xls", ".xlsx"]:
            df_landings = pd.read_excel(path_landings, sheet_name=None)
            df_gear = pd.read_excel(path_gear, sheet_name=None)
            monthly_df = df_landings[list(df_landings.keys())[0]]
            gear_df = df_gear[list(df_gear.keys())[0]]
        else:
            print(f"❌ Unsupported file format: {filename}")
            continue

        # ✅ Clean species names in both datasets
        monthly_df['Species'] = monthly_df['Species'].astype(str).str.replace(r"[\\_\-]", "", regex=True).str.replace("/", "_").str.strip()
        gear_df['Species'] = gear_df['Species'].astype(str).str.replace(r"[\\_\-]", "", regex=True).str.replace("/", "_").str.strip()
        gear_df['Species'] = gear_df['Species'].str.replace("/", "_").str.strip()


        # ✅ Save cleaned data back to the original file
        if ext == ".csv":
            monthly_df.to_csv(path_landings, index=False)
            gear_df.to_csv(path_gear, index=False)
        else:  # For Excel, overwrite the same sheet
            with pd.ExcelWriter(path_landings, engine="openpyxl", mode="w") as writer:
                monthly_df.to_excel(writer, index=False, sheet_name="Sheet1")
            with pd.ExcelWriter(path_gear, engine="openpyxl", mode="w") as writer:
                gear_df.to_excel(writer, index=False, sheet_name="Sheet1")

        print(f"✅ Cleaned and saved: {filename}")

    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")


Processing: Tenggiri.csv
✅ Cleaned and saved: Tenggiri.csv
Processing: Sebelah.csv
✅ Cleaned and saved: Sebelah.csv
Processing: Yu.csv
✅ Cleaned and saved: Yu.csv
Processing: Bawalputih.csv
✅ Cleaned and saved: Bawalputih.csv
Processing: Alu-Alu_Kacang-Kacang.csv
✅ Cleaned and saved: Alu-Alu_Kacang-Kacang.csv
Processing: Bijinangka.csv
✅ Cleaned and saved: Bijinangka.csv
Processing: Kayu_Tongkol_Ayahitam.csv
✅ Cleaned and saved: Kayu_Tongkol_Ayahitam.csv
Processing: Gelama_Tengkerong.csv
✅ Cleaned and saved: Gelama_Tengkerong.csv
Processing: Bawalhitam.csv
✅ Cleaned and saved: Bawalhitam.csv
Processing: Kerisi.csv
✅ Cleaned and saved: Kerisi.csv
Processing: Siakap.csv
✅ Cleaned and saved: Siakap.csv
Processing: Kerisibali.csv
✅ Cleaned and saved: Kerisibali.csv
Processing: Selar.csv
✅ Cleaned and saved: Selar.csv
Processing: Ketamlaut.csv
✅ Cleaned and saved: Ketamlaut.csv
Processing: Selarkuning.csv
✅ Cleaned and saved: Selarkuning.csv


## Merge Them Together

In [ ]:
for filename in common_files:
    print(f"Processing: {filename}")

    path_landings = os.path.join(folder_landings, filename)
    path_gear = os.path.join(folder_gear_imputed, filename)

    ext = os.path.splitext(filename)[1].lower()

    try:
        if ext == ".csv":
            monthly_df = pd.read_csv(path_landings)
            gear_df = pd.read_csv(path_gear)
        elif ext in [".xls", ".xlsx"]:
            df_landings = pd.read_excel(path_landings, sheet_name=None)
            df_gear = pd.read_excel(path_gear, sheet_name=None)
            monthly_df = df_landings[list(df_landings.keys())[0]]
            gear_df = df_gear[list(df_gear.keys())[0]]
        else:
            print(f"❌ Unsupported file format: {filename}")
            continue

        # Clean species names and save them back
        monthly_df['Species'] = monthly_df['Species'].astype(str)\
            .str.replace(r"[\\_\-]", "", regex=True)\
            .str.replace("/", "_")\
            .str.strip()

        gear_df['Species'] = gear_df['Species'].astype(str)\
            .str.replace(r"[\\_\-]", "", regex=True)\
            .str.replace("/", "_")\
            .str.strip()

        # Clean monthly landing data
        monthly_df = monthly_df.rename(columns={"Species": "species", "Year": "year", "Month": "month"})
        monthly_df = monthly_df.melt(id_vars=["species", "year", "month"], var_name="state", value_name="landings")

        # Replace NaN in landings with 0 (keep all rows)
        monthly_df["landings"] = monthly_df["landings"].fillna(0)

        # Clean gear composition data
        gear_df = gear_df.rename(columns={"Species": "species", "Year": "year", "Location": "location"})

        # Melt gear_df from wide to long
        gear_df = gear_df.melt(id_vars=["species", "year", "location"], var_name="gear_type", value_name="proportion")

        # Convert proportions from percent if > 1
        if gear_df["proportion"].max() > 1:
            gear_df["proportion"] = gear_df["proportion"] / 100.0

        # Helper: match gear data by location/region/Malaysia
        def match_gear(row):
            species, year, state = row['species'], row['year'], row['state']
            subset = gear_df[(gear_df['species'] == species) & (gear_df['year'] == year)]

            if state in subset["location"].values:
                return subset[subset["location"] == state]

            region = state_to_region.get(state)
            if region and region in subset["location"].values:
                return subset[subset["location"] == region]

            if "Malaysia" in subset["location"].values:
                return subset[subset["location"] == "Malaysia"]

            return pd.DataFrame()

        # Track missing gear data
        missing_gear_summary = defaultdict(int)
        disaggregated_rows = set()

        # Spatial disaggregation
        for _, row in monthly_df.iterrows():
            gear_comp = match_gear(row)
            if gear_comp.empty:
                key = (row['species'], row['year'], row['state'])
                missing_gear_summary[key] += 1
                continue

            for _, g_row in gear_comp.iterrows():
                new_row = (
                    row["species"],
                    row["month"],
                    row["state"],
                    g_row["gear_type"],
                    round(row["landings"] * g_row["proportion"], 8),
                    row["year"],
                )
                disaggregated_rows.add(new_row)

        # Convert to DataFrame and sort
        disaggregated_df = pd.DataFrame(
            list(disaggregated_rows),
            columns=["species", "month", "state", "gear_type","landings", "year"]
        )

        disaggregated_df = disaggregated_df.sort_values(by=["year", "month", "state", "gear_type"])

        if not disaggregated_df.empty:
            out_path = os.path.join(folder_output, f"{filename.replace(ext, '.csv')}")
            disaggregated_df.to_csv(out_path, index=False)
            print(f"✔️ Saved: {out_path}")
        else:
            print(f"⚠️ No disaggregated data for: {filename}")

        # Print summary of missing gear data
        if missing_gear_summary:
            print("⚠️ Summary of missing gear data:")
            for key, count in missing_gear_summary.items():
                print(f"   {key[0]} - {key[1]} - {key[2]}: {count} record(s) skipped")

    except Exception as e:
        print(f"❌ Error processing {filename}: {e}")


Processing: Tenggiri.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Tenggiri.csv
Processing: Sebelah.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Sebelah.csv
Processing: Yu.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Yu.csv
Processing: Bawalputih.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Bawalputih.csv
Processing: Alu-Alu_Kacang-Kacang.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Alu-Alu_Kacang-Kacang.csv
Processing: Bijinangka.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Bijinangka.csv
Processing: Kayu_Tongkol_Ayahitam.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Kayu_Tongkol_Ayahitam.csv
Processing: Gelama_Tengkerong.csv
✔️ Saved: /content/drive/My Drive/FYP_Fish/monthly_landings_state_gear/Gelama_Tengkerong.csv
Processing: Bawalhitam.csv
✔️ Saved: /content/drive/My Drive/FYP

## Print number of rows for each csv

In [ ]:
def print_csv_row_counts(folder_path):
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    df = pd.read_csv(file_path)
                    print(f"{file}: {len(df)} rows")
                except Exception as e:
                    print(f"❌ Failed to read {file}: {e}")


In [ ]:
print_csv_row_counts(folder_output)

Siakap.csv: 32760 rows
Tenggiri.csv: 32760 rows
Bawalhitam.csv: 32760 rows
Alu-Alu_Kacang-Kacang.csv: 32760 rows
Selarkuning.csv: 32760 rows
Kerisi.csv: 32760 rows
Bawalputih.csv: 32760 rows
Kayu_Tongkol_Ayahitam.csv: 32760 rows
Ketamlaut.csv: 32760 rows
Selar.csv: 32760 rows
Yu.csv: 32760 rows
Gelama_Tengkerong.csv: 32760 rows
Sebelah.csv: 32760 rows
Bijinangka.csv: 32760 rows
Kerisibali.csv: 32760 rows
